In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
#!/usr/bin/env python3
# ============================================================
# Ready-to-run: Protocol-Valid Fair Auditing of High-Confidence
# Failures under Distribution Shift (TABULAR)
#
# TableShift-like UCI Diabetic Readmission (UCI id=296)
# Common pipeline aligned to the TEXT script:
#   - Base classifier + temperature scaling
#   - Monitoring target z = 1[err & conf >= TAU_TARGET]
#   - Monitoring slice conf >= HIGH_CONF_REGION
#   - Risk scorers: entropy, energy, RED-like LR, TCP, TCP-M,
#     KOF(+stability), tcp_gate_tau, tcp_m_gate_tau,
#     kof_stab_gate_tau, random_gate_tau
#   - TableShift-like ID/OOD split via admission_source_id == 7
#   - Mixture rho + synthetic drift stress tests
#   - Fair allocation policies: global / parity / bounded-topk(delta)
#   - Robust reporting: per-group counts, worst-group metrics,
#     bootstrap CI, Pareto, selection-change confirmation
#
# This script also keeps the same drift ablation strategy as the
# previous diabetes script and runs all experiment arms in one go:
#   - full
#   - abl_no_swap
#   - abl_no_perm
#
# Sensitive attribute (eval-only): gender (Male=1, Female=0),
# Unknown/Invalid dropped.
#
# Outputs (OUTDIR):
#   tables/: raw_* + summary_* (mean/std + bootstrap CI)
#   figs/: core curves + mitigation + Pareto
#   report.md + OUTDIR.zip
# ============================================================

import os
import sys
import json
import math
import random
import zipfile
import argparse
import gc
import subprocess
import warnings
from dataclasses import dataclass, asdict
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)


# -----------------------------
# Install deps (Colab/Kaggle safe)
# -----------------------------
def pip_install(pkgs: List[str]):
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install"] + pkgs)


pip_install([
    "numpy",
    "pandas",
    "tqdm",
    "scikit-learn>=1.2",
    "matplotlib",
    "tabulate",
    "ucimlrepo>=0.0.7",
])

from tqdm import tqdm
import matplotlib.pyplot as plt
import importlib.metadata as md

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.isotonic import IsotonicRegression

from ucimlrepo import fetch_ucirepo


DATASET_NAME = "uci_diabetic_readmission_296"
SENSITIVE_COL = "gender"
GENDER_KEEP = ("Male", "Female")

DEFAULT_ID_TEST_SIZE = 0.10
DEFAULT_OOD_VAL_SIZE = 0.10
DEFAULT_ID_VAL_SIZE = 0.10
DEFAULT_RANDOM_STATE = 264738

NUMERIC_COLS_HINT = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
]


# -----------------------------
# Args
# -----------------------------
@dataclass
class Args:
    DATA_DIR: str = "./uci_cache"
    OUTDIR: str = "./out_tableshift_diabetes_common_monitor"
    USE_KAGGLE_DIR: bool = True

    DEVICE: str = "auto"
    DETERMINISTIC: bool = True
    NUM_WORKERS: int = 0

    SEEDS: List[int] = None
    MAX_TRAIN: int = 60000
    MAX_VAL: int = 25000
    MAX_ID_TEST: int = 30000
    MAX_OOD_VAL: int = 25000
    MAX_OOD_TEST: int = 30000
    N_TARGET: int = 20000

    BATCH_SIZE: int = 512
    HIDDEN: int = 256
    DROPOUT: float = 0.10

    EPOCHS_CLS: int = 10
    LR_CLS: float = 2e-3
    WD: float = 1e-4
    EARLY_STOP_METRIC: str = "nll"
    EARLY_STOP_PATIENCE: int = 2
    EARLY_STOP_MIN_DELTA: float = 1e-4

    USE_TEMP_SCALING: bool = True
    TS_MAX_ITERS: int = 250

    HIGH_CONF_REGION: float = 0.80
    TAU_TARGET: float = 0.90
    ALERT_BUDGETS: List[float] = None

    RHO_VALUES: List[float] = None

    SWAP_COL_PREFS: List[str] = None
    PERM_COL_PREFS: List[str] = None
    DRIFT_BASE: float = 0.20
    DRIFT_GAIN: float = 0.80
    SWAP_TOPK: int = 60
    PERM_TOPK: int = 80
    DRIFT_ON_ID_AT_HIGH_RHO: bool = True
    ID_DRIFT_SCALE: float = 0.25

    TAU_LIST: List[float] = None
    KOF_HIDDEN: int = 256
    EPOCHS_DETECT: int = 4
    LR_DETECT: float = 2e-3

    EPOCHS_TCPM: int = 8
    TCPM_EARLY_STOP_PATIENCE: int = 2
    TCPM_EARLY_STOP_MIN_DELTA: float = 1e-4
    TCPM_TRAIN_ON_TAU_SLICE: bool = True
    TCPM_USE_POS_WEIGHT: bool = True
    TCPM_APPLY_TAU_GATE_AT_INFERENCE: bool = True

    USE_STABILITY: bool = True
    STAB_K: int = 4
    STAB_PERT_PROB: float = 0.35
    STAB_CONT_NOISE_STD: float = 0.12
    STAB_MAX_SAMPLES_FOR_FEAT: int = 40000

    ENABLE_FAIRNESS_MITIGATION: bool = True
    FAIRNESS_POLICIES: List[str] = None
    FAIR_TOPK_DELTAS: List[float] = None
    MITIGATE_MODELS: List[str] = None

    GROUP_MIN_N_LIST: List[int] = None
    BOOTSTRAP_N: int = 1200
    BOOTSTRAP_ALPHA: float = 0.05


@dataclass
class ExperimentConfig:
    name: str
    use_swap_drift: bool = True
    use_perm_drift: bool = True


def make_args() -> Args:
    a = Args()
    a.SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61]
    a.RHO_VALUES = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    a.TAU_LIST = [0.70, 0.80, 0.90, 0.95]
    a.ALERT_BUDGETS = [0.05, 0.10]

    a.FAIRNESS_POLICIES = ["global", "parity", "bounded"]
    a.FAIR_TOPK_DELTAS = [0.02, 0.05]
    a.MITIGATE_MODELS = ["tcp_m_gate_tau", "kof_stab_gate_tau", "kof_stab", "tcp_gate_tau"]

    a.GROUP_MIN_N_LIST = [1, 200, 500]

    a.SWAP_COL_PREFS = [
        "diag_1", "diag_2", "diag_3", "medical_specialty",
        "admission_type_id", "discharge_disposition_id",
        "admission_source_id", "race"
    ]
    a.PERM_COL_PREFS = [
        "diag_2", "diag_1", "diag_3", "race",
        "discharge_disposition_id", "medical_specialty",
        "admission_type_id", "admission_source_id"
    ]

    if a.USE_KAGGLE_DIR and os.path.exists("/kaggle/working"):
        a.OUTDIR = "/kaggle/working/out_tableshift_diabetes_common_monitor"
        a.DATA_DIR = "/kaggle/working/uci_cache"
    return a


args = make_args()


def experiments() -> List[ExperimentConfig]:
    return [
        ExperimentConfig(name="full", use_swap_drift=True, use_perm_drift=True),
        ExperimentConfig(name="abl_no_swap", use_swap_drift=False, use_perm_drift=True),
        ExperimentConfig(name="abl_no_perm", use_swap_drift=True, use_perm_drift=False),
    ]


# -----------------------------
# Utilities
# -----------------------------
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def set_determinism(enable: bool):
    if not enable:
        return
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass


def get_device(name: str):
    if name == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(name)



def _normalize_binary_scores(y_true, y_score):
    y_true = np.asarray(y_true).astype(int).reshape(-1)
    ys = np.asarray(y_score)

    if ys.ndim == 0:
        ys = np.full(len(y_true), float(ys), dtype=float)
    elif ys.ndim == 1:
        ys = ys.astype(float).reshape(-1)
    elif ys.ndim == 2:
        if ys.shape[0] == len(y_true) and ys.shape[1] >= 2:
            ys = ys[:, 1].astype(float).reshape(-1)
        elif ys.shape[1] == len(y_true) and ys.shape[0] >= 2:
            ys = ys[1, :].astype(float).reshape(-1)
        else:
            ys = ys.reshape(-1).astype(float)
    else:
        ys = ys.reshape(-1).astype(float)

    if len(y_true) != len(ys):
        raise ValueError(
            f"Label/score length mismatch after normalization: "
            f"y_true.shape={np.asarray(y_true).shape}, y_score.shape={np.asarray(y_score).shape}, "
            f"normalized={len(y_true)} vs {len(ys)}"
        )
    return y_true, ys


class ConstantProbModel:
    """Fallback model for degenerate single-class training targets."""
    def __init__(self, p_pos: float):
        self.p_pos = float(np.clip(p_pos, 0.0, 1.0))

    def predict_proba(self, X):
        X = np.asarray(X)
        n = int(X.shape[0]) if X.ndim > 0 else 1
        p1 = np.full(n, self.p_pos, dtype=np.float32)
        p0 = 1.0 - p1
        return np.column_stack([p0, p1])


def safe_auc(y_true, y_score) -> float:
    y_true = np.asarray(y_true).astype(int).reshape(-1)
    y_score = np.asarray(y_score).astype(float).reshape(-1)
    if len(y_true) == 0 or len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_score))


def p_drift_from_rho(rho: float) -> float:
    return float(np.clip(args.DRIFT_BASE + args.DRIFT_GAIN * float(rho), 0.0, 1.0))


def ensure_outdirs(outdir: str):
    os.makedirs(outdir, exist_ok=True)
    os.makedirs(os.path.join(outdir, "tables"), exist_ok=True)
    os.makedirs(os.path.join(outdir, "figs"), exist_ok=True)

# -----------------------------
# Dataset / shift statistics reporting (KBS reviewer comment #2)
# -----------------------------
def compute_split_stats(name: str, rho, y: np.ndarray, g: np.ndarray,
                         z: Optional[np.ndarray] = None, high_mask: Optional[np.ndarray] = None) -> Dict[str, object]:
    """
    Records dataset sizes, group distributions, class imbalance,
    and (when available) high-confidence-failure / positive-target counts
    for a given split or shift condition (rho).
    """
    y = np.asarray(y).reshape(-1).astype(np.int64)
    g = np.asarray(g).reshape(-1).astype(np.int64)
    n_total = int(len(y))

    n_group0 = int((g == 0).sum())
    n_group1 = int((g == 1).sum())
    n_pos = int(y.sum())
    n_neg = n_total - n_pos

    stats: Dict[str, object] = {
        "split": name,
        "rho": (float(rho) if rho is not None else float("nan")),
        "n_total": n_total,
        "n_group0": n_group0,
        "n_group1": n_group1,
        "group0_frac": float(n_group0 / n_total) if n_total > 0 else float("nan"),
        "group1_frac": float(n_group1 / n_total) if n_total > 0 else float("nan"),
        "n_pos_target": n_pos,
        "n_neg_target": int(n_neg),
        "pos_rate": float(n_pos / n_total) if n_total > 0 else float("nan"),
        "class_imbalance_ratio": float(n_neg / max(n_pos, 1)),
        "n_pos_group0": int(y[g == 0].sum()) if n_group0 > 0 else 0,
        "n_pos_group1": int(y[g == 1].sum()) if n_group1 > 0 else 0,
    }

    if high_mask is not None:
        high_mask = np.asarray(high_mask).reshape(-1).astype(bool)
        n_high = int(high_mask.sum())
        stats["n_high_conf"] = n_high
        stats["high_conf_frac"] = float(n_high / n_total) if n_total > 0 else float("nan")
        stats["n_high_conf_group0"] = int((high_mask & (g == 0)).sum())
        stats["n_high_conf_group1"] = int((high_mask & (g == 1)).sum())
    else:
        n_high = None

    if z is not None:
        z = np.asarray(z).reshape(-1).astype(np.int64)
        stats["n_high_conf_failures"] = int(z.sum())
        stats["failure_rate_in_high_conf"] = float(z.mean()) if len(z) > 0 else float("nan")
        if high_mask is not None and n_high is not None and n_high > 0:
            z_high = z[high_mask]
            g_high = g[high_mask]
            stats["n_failures_group0"] = int(z_high[g_high == 0].sum())
            stats["n_failures_group1"] = int(z_high[g_high == 1].sum())
            n_hg0 = int((g_high == 0).sum())
            n_hg1 = int((g_high == 1).sum())
            stats["failure_rate_group0"] = float(stats["n_failures_group0"] / n_hg0) if n_hg0 > 0 else float("nan")
            stats["failure_rate_group1"] = float(stats["n_failures_group1"] / n_hg1) if n_hg1 > 0 else float("nan")

    return stats


def dump_dataset_statistics(all_stats: List[Dict[str, object]], outdir: str) -> pd.DataFrame:
    df = pd.DataFrame(all_stats)
    df.to_csv(os.path.join(outdir, "tables", "dataset_shift_statistics.csv"), index=False)
    return df




def batch_iterable(n: int, bs: int):
    for i in range(0, n, bs):
        yield i, min(n, i + bs)


def dump_versions() -> Dict[str, str]:
    pkgs = ["ucimlrepo", "scikit-learn", "torch", "pandas", "numpy", "matplotlib"]
    out = {}
    for p in pkgs:
        try:
            out[p] = md.version(p)
        except Exception:
            out[p] = "unknown"
    return out


def mean_std_ci_over_seeds(
    df: pd.DataFrame,
    group_cols: List[str],
    metric_cols: List[str],
    seed_col: str = "seed",
    n_boot: int = 1000,
    alpha: float = 0.05,
) -> pd.DataFrame:
    out_rows = []
    base = df.copy()
    base_seed = base.groupby(group_cols + [seed_col], as_index=False)[metric_cols].mean()

    nb = int(n_boot) if (n_boot is not None and int(n_boot) > 0) else 0
    rng = np.random.default_rng(12345)

    for keys, sub in base_seed.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = {c: v for c, v in zip(group_cols, keys)}

        for m in metric_cols:
            vals = sub[m].to_numpy(dtype=float)
            row[f"{m}_mean"] = float(np.nanmean(vals)) if np.isfinite(vals).any() else float("nan")
            row[f"{m}_std"] = float(np.nanstd(vals, ddof=1)) if np.isfinite(vals).sum() > 1 else float("nan")

            seed_vals = sub.groupby(seed_col)[m].mean().to_numpy(dtype=float)
            finite = seed_vals[np.isfinite(seed_vals)]
            s_eff = int(len(finite))
            row[f"{m}_n_seed_finite"] = s_eff

            if nb < 1 or s_eff < 2:
                row[f"{m}_ci_low"] = float("nan")
                row[f"{m}_ci_high"] = float("nan")
            else:
                boot = np.empty(nb, dtype=float)
                for bi in range(nb):
                    idx = rng.integers(0, s_eff, size=s_eff)
                    boot[bi] = float(np.mean(finite[idx]))
                row[f"{m}_ci_low"] = float(np.quantile(boot, alpha / 2))
                row[f"{m}_ci_high"] = float(np.quantile(boot, 1 - alpha / 2))
        out_rows.append(row)

    return pd.DataFrame(out_rows)


# -----------------------------
# UCI loading + TableShift-like split
# -----------------------------
def _maybe_subsample_df(X: pd.DataFrame, y: np.ndarray, g: np.ndarray, max_n: Optional[int], seed: int):
    if (max_n is None) or (max_n <= 0) or (len(X) <= max_n):
        return X.reset_index(drop=True), y.reshape(-1), g.reshape(-1)
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(X), size=int(max_n), replace=False)
    idx = np.sort(idx)
    return X.iloc[idx].reset_index(drop=True), y[idx].reshape(-1), g[idx].reshape(-1)


def _standardize_strings(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in out.columns:
        if out[c].dtype == object:
            out[c] = out[c].replace("?", np.nan)
    return out


def _gender_to_binary(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip().str.lower()
    out = pd.Series(-1, index=series.index, dtype="int64")
    out.loc[s == "male"] = 1
    out.loc[s == "female"] = 0
    return out


def _collapse_rare_categories(train_df: pd.DataFrame, other_dfs: List[pd.DataFrame], cols: List[str], min_freq: float):
    if min_freq is None or float(min_freq) <= 0:
        return
    for c in cols:
        if c not in train_df.columns:
            continue
        vc = train_df[c].astype(str).value_counts(normalize=True, dropna=False)
        rare = set(vc[vc < float(min_freq)].index.tolist())
        if not rare:
            continue
        train_df[c] = train_df[c].astype(str).map(lambda v: "__RARE__" if v in rare else v)
        for df in other_dfs:
            df[c] = df[c].astype(str).map(lambda v: "__RARE__" if v in rare else v)


def load_uci_diabetes_readmission_splits(seed: int):
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)
        ds = fetch_ucirepo(id=296)
    X = ds.data.features.copy()
    ydf = ds.data.targets.copy()
    if ydf.shape[1] != 1:
        raise RuntimeError(f"Expected 1 target column, got: {list(ydf.columns)}")
    tgt_col = ydf.columns[0]

    df = X.join(ydf)
    df = _standardize_strings(df)

    if "race" not in df.columns:
        raise RuntimeError("Expected 'race' column in UCI dataset.")
    df = df[df["race"].notna()].copy()

    if SENSITIVE_COL not in df.columns:
        raise RuntimeError(f"Expected '{SENSITIVE_COL}' column in UCI dataset.")
    g_raw = _gender_to_binary(df[SENSITIVE_COL])
    df = df[g_raw >= 0].copy()

    if "admission_source_id" not in df.columns:
        raise RuntimeError("Expected 'admission_source_id' column in UCI dataset.")
    dom = pd.to_numeric(df["admission_source_id"], errors="coerce")
    df = df.loc[dom.notna()].copy()
    dom = pd.to_numeric(df["admission_source_id"], errors="coerce").astype(int).to_numpy()

    g = _gender_to_binary(df[SENSITIVE_COL]).to_numpy(dtype=np.int64)
    y_series = df[tgt_col].astype(str).str.strip().str.upper()
    y = (y_series != "NO").astype(np.int64).to_numpy()

    drop_cols = [SENSITIVE_COL]
    for c in ["encounter_id", "patient_nbr", tgt_col]:
        if c in df.columns:
            drop_cols.append(c)
    dfX = df.drop(columns=[c for c in drop_cols if c in df.columns]).reset_index(drop=True)

    for c in NUMERIC_COLS_HINT:
        if c in dfX.columns:
            dfX[c] = pd.to_numeric(dfX[c], errors="coerce")

    idx_all = np.arange(len(dfX))
    idx_ood = idx_all[dom == 7]
    idx_id = idx_all[dom != 7]
    if len(idx_ood) == 0 or len(idx_id) == 0:
        raise RuntimeError("TableShift-like split failed: missing ID or OOD rows.")

    rng = np.random.default_rng(seed + DEFAULT_RANDOM_STATE)
    rng.shuffle(idx_id)
    rng.shuffle(idx_ood)

    n_total = len(dfX)
    n_id_val = int(round(DEFAULT_ID_VAL_SIZE * n_total))
    n_id_test = int(round(DEFAULT_ID_TEST_SIZE * n_total))
    n_ood_val = int(round(DEFAULT_OOD_VAL_SIZE * n_total))

    n_id_test = min(n_id_test, max(1, len(idx_id) - 2))
    n_id_val = min(n_id_val, max(1, len(idx_id) - n_id_test - 1))
    if n_id_test + n_id_val >= len(idx_id):
        n_id_val = max(1, len(idx_id) - n_id_test - 1)

    id_test_idx = idx_id[:n_id_test]
    id_val_idx = idx_id[n_id_test:n_id_test + n_id_val]
    id_train_idx = idx_id[n_id_test + n_id_val:]

    n_ood_val = min(n_ood_val, max(1, len(idx_ood) - 1))
    ood_val_idx = idx_ood[:n_ood_val]
    ood_test_idx = idx_ood[n_ood_val:]
    if len(ood_test_idx) == 0:
        ood_test_idx = idx_id[:max(1, min(len(idx_id), n_id_test))]

    def take(ix):
        return {
            "X": dfX.iloc[ix].reset_index(drop=True),
            "y": y[ix].astype(np.int64).reshape(-1),
            "g": g[ix].astype(np.int64).reshape(-1),
        }

    splits = {
        "train": take(id_train_idx),
        "validation": take(id_val_idx),
        "id_test": take(id_test_idx),
        "ood_validation": take(ood_val_idx),
        "ood_test": take(ood_test_idx),
    }

    trainX = splits["train"]["X"]
    numeric_cols = [c for c in NUMERIC_COLS_HINT if c in trainX.columns]
    med = {}
    if numeric_cols:
        med = {c: float(np.nanmedian(trainX[c].to_numpy(dtype=float))) for c in numeric_cols}

    for sp in splits.values():
        Xsp = sp["X"]
        for c in numeric_cols:
            Xsp[c] = pd.to_numeric(Xsp[c], errors="coerce").fillna(med[c])
        for c in Xsp.columns:
            if c in numeric_cols:
                continue
            Xsp[c] = Xsp[c].astype(object).where(Xsp[c].notna(), "MISSING")

    _collapse_rare_categories(
        splits["train"]["X"],
        [splits["validation"]["X"], splits["id_test"]["X"], splits["ood_validation"]["X"], splits["ood_test"]["X"]],
        cols=["diag_1", "diag_2", "diag_3"],
        min_freq=0.01,
    )

    split_max = {
        "train": args.MAX_TRAIN,
        "validation": args.MAX_VAL,
        "id_test": args.MAX_ID_TEST,
        "ood_validation": args.MAX_OOD_VAL,
        "ood_test": args.MAX_OOD_TEST,
    }
    offsets = {"train": 1, "validation": 2, "id_test": 3, "ood_validation": 4, "ood_test": 5}
    for name, sp in splits.items():
        Xs, ys, gs = _maybe_subsample_df(sp["X"], sp["y"], sp["g"], split_max.get(name, None), seed + offsets[name])
        sp["X"], sp["y"], sp["g"] = Xs, ys, gs

    splits["meta"] = {
        "dataset": DATASET_NAME,
        "uci_id": 296,
        "ood_definition": "admission_source_id == 7",
        "sensitive_col_eval_only": SENSITIVE_COL,
        "gender_keep": list(GENDER_KEEP),
    }
    return splits


# -----------------------------
# Drift stress tests
# -----------------------------
def pick_existing_col(preferences: List[str], X_df: pd.DataFrame, exclude: set = None) -> str:
    exclude = exclude or set()
    for c in preferences:
        if c in X_df.columns and c not in exclude:
            return c
    best, best_u = None, -1
    for c in X_df.columns:
        if c in exclude:
            continue
        u = int(X_df[c].astype(str).nunique(dropna=True))
        if 10 <= u <= 2000 and u > best_u:
            best_u = u
            best = c
    return best if best is not None else X_df.columns[0]


def build_swap_mapping_label_informed(X_df: pd.DataFrame, y: np.ndarray, col: str, top_k: int) -> Dict[str, str]:
    df = pd.DataFrame({"v": X_df[col].astype(str), "y": np.asarray(y).reshape(-1).astype(int)})
    grp = df.groupby("v")["y"].mean().sort_values()
    if len(grp) < 2:
        return {}
    k = min(int(top_k), len(grp) // 2)
    low = grp.index[:k].tolist()
    high = grp.index[-k:].tolist()
    m = {}
    for a, b in zip(low, high):
        m[a] = b
        m[b] = a
    return m


def build_perm_mapping_label_free(series: pd.Series, top_k: int, seed: int) -> Dict[str, str]:
    s = series.astype(str)
    freq = s.value_counts()
    keys = freq.index[:min(int(top_k), len(freq))].tolist()
    rng = np.random.default_rng(seed)
    perm = keys.copy()
    rng.shuffle(perm)
    return dict(zip(keys, perm))


def apply_mapping_drift(X_df: pd.DataFrame, col: str, mapping: Dict[str, str], prob: float, seed: int) -> Tuple[pd.DataFrame, np.ndarray]:
    prob = float(np.clip(prob, 0.0, 1.0))
    if prob <= 0 or not mapping or (col not in X_df.columns):
        return X_df.copy(), np.zeros(len(X_df), dtype=np.int8)
    rng = np.random.default_rng(seed)
    X = X_df.copy()
    s = X[col].astype(str)
    in_map = s.isin(mapping.keys()).to_numpy()
    flip = rng.random(len(X)) < prob
    change = in_map & flip
    if change.any():
        s2 = s.copy()
        s2.loc[change] = s2.loc[change].map(mapping).fillna(s2.loc[change])
        X[col] = s2.values
    return X, change.astype(np.int8)


def apply_combined_drift(
    cfg: ExperimentConfig,
    X_df: pd.DataFrame,
    rho: float,
    seed: int,
    swap_col: str,
    perm_col: str,
    swap_map: Dict[str, str],
    perm_map: Dict[str, str],
    p_override: Optional[float] = None,
) -> Tuple[pd.DataFrame, Dict[str, np.ndarray]]:
    p = p_drift_from_rho(rho) if p_override is None else float(np.clip(p_override, 0.0, 1.0))
    if cfg.use_swap_drift:
        X1, m_swap = apply_mapping_drift(X_df, swap_col, swap_map, prob=p, seed=seed + 1)
    else:
        X1, m_swap = X_df.copy(), np.zeros(len(X_df), dtype=np.int8)
    if cfg.use_perm_drift:
        X2, m_perm = apply_mapping_drift(X1, perm_col, perm_map, prob=p, seed=seed + 2)
    else:
        X2, m_perm = X1, np.zeros(len(X_df), dtype=np.int8)
    return X2, {"swap": m_swap, "perm": m_perm, "any": ((m_swap + m_perm) > 0).astype(np.int8)}


# -----------------------------
# Preprocessing
# -----------------------------
def infer_cat_cont_cols(X_df: pd.DataFrame, force_cat_cols: List[str]) -> Tuple[List[str], List[str]]:
    default_cont = set(NUMERIC_COLS_HINT)
    cont = [c for c in X_df.columns if c in default_cont]
    if len(cont) == 0:
        cont = []
        for c in X_df.columns:
            s = X_df[c]
            if pd.api.types.is_numeric_dtype(s) and s.nunique(dropna=True) > 100:
                cont.append(c)
    cat = [c for c in X_df.columns if c not in cont]
    for dc in force_cat_cols:
        if dc in cont:
            cont.remove(dc)
            cat.append(dc)
    return cat, cont


def fit_encoders(X_train: pd.DataFrame, cat_cols: List[str], cont_cols: List[str]):
    cat_maps, cat_cards = {}, {}
    for c in cat_cols:
        vals = pd.Series(X_train[c].astype(str).unique())
        mapping = {v: (i + 1) for i, v in enumerate(vals.tolist())}
        cat_maps[c] = mapping
        cat_cards[c] = len(mapping) + 1
    if len(cont_cols):
        cont_mean = X_train[cont_cols].astype(np.float32).mean(axis=0).to_numpy().astype(np.float32)
        cont_std = X_train[cont_cols].astype(np.float32).std(axis=0).replace(0.0, 1.0).to_numpy().astype(np.float32)
    else:
        cont_mean = np.array([], dtype=np.float32)
        cont_std = np.array([], dtype=np.float32)
    return cat_maps, cat_cards, cont_mean, cont_std


def encode_split(
    X_df: pd.DataFrame,
    y: np.ndarray,
    cat_cols: List[str],
    cont_cols: List[str],
    cat_maps: Dict[str, Dict[str, int]],
    cont_mean: np.ndarray,
    cont_std: np.ndarray,
):
    y = np.asarray(y).reshape(-1).astype(np.int64)
    if len(cat_cols):
        Xc = np.zeros((len(X_df), len(cat_cols)), dtype=np.int64)
        for j, c in enumerate(cat_cols):
            mapping = cat_maps[c]
            Xc[:, j] = X_df[c].astype(str).map(lambda v: mapping.get(v, 0)).to_numpy(dtype=np.int64)
    else:
        Xc = np.zeros((len(X_df), 0), dtype=np.int64)

    if len(cont_cols):
        Xraw = X_df[cont_cols].astype(np.float32).to_numpy()
        Xn = ((Xraw - cont_mean) / cont_std).astype(np.float32)
    else:
        Xn = np.zeros((len(X_df), 0), dtype=np.float32)
    return Xc, Xn, y


class TabDataset(Dataset):
    def __init__(self, Xc, Xn, y):
        self.Xc = torch.from_numpy(np.asarray(Xc))
        self.Xn = torch.from_numpy(np.asarray(Xn))
        self.y = torch.from_numpy(np.asarray(y).reshape(-1).astype(np.int64)).long()

    def __len__(self):
        return int(self.y.numel())

    def __getitem__(self, i):
        return self.Xc[i], self.Xn[i], self.y[i]


# -----------------------------
# Base model
# -----------------------------
def emb_dim(card: int) -> int:
    return int(min(64, max(4, round(math.sqrt(card) * 2))))


class TabMLP(nn.Module):
    def __init__(self, cat_cards: Dict[str, int], n_cont: int, hidden: int, dropout: float):
        super().__init__()
        self.cat_cols = list(cat_cards.keys())
        self.embs = nn.ModuleList()
        emb_dims = []
        for c in self.cat_cols:
            card = cat_cards[c]
            d = emb_dim(card)
            emb_dims.append(d)
            self.embs.append(nn.Embedding(card, d))
        in_dim = sum(emb_dims) + n_cont
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.out = nn.Linear(hidden, 2)

    def forward(self, Xc, Xn):
        parts = [emb(Xc[:, j]) for j, emb in enumerate(self.embs)]
        if len(parts):
            z = torch.cat(parts, dim=-1)
            if Xn.numel():
                z = torch.cat([z, Xn], dim=-1)
        else:
            z = Xn
        h = F.relu(self.fc1(z), inplace=True)
        h = self.drop(h)
        h = F.relu(self.fc2(h), inplace=True)
        h = self.drop(h)
        logits = self.out(h)
        return h, logits


def probs_from_logits(logits):
    return F.softmax(logits, dim=-1)


def conf_from_logits(logits):
    return probs_from_logits(logits).max(dim=-1).values


def entropy_from_logits(logits):
    p = probs_from_logits(logits).clamp_min(1e-8)
    return -(p * p.log()).sum(dim=-1)


def margin_from_logits(logits):
    top2 = torch.topk(logits, k=2, dim=-1).values
    return top2[:, 0] - top2[:, 1]


def energy_from_logits(logits):
    return -torch.logsumexp(logits, dim=-1)


class TempScaler(nn.Module):
    def __init__(self, init_T: float = 1.0):
        super().__init__()
        self.logT = nn.Parameter(torch.tensor([math.log(init_T)], dtype=torch.float32))

    def forward(self, logits):
        T = torch.exp(self.logT).clamp(1e-3, 1e3)
        return logits / T


@torch.no_grad()
def eval_cls(model, loader, device, temp_scaler: Optional[TempScaler] = None):
    model.eval()
    if temp_scaler is not None:
        temp_scaler.eval()
    tot, correct, nll_sum = 0, 0, 0.0
    for Xc, Xn, y in loader:
        Xc, Xn, y = Xc.to(device), Xn.to(device), y.to(device)
        _, logits = model(Xc, Xn)
        if temp_scaler is not None:
            logits = temp_scaler(logits)
        nll_sum += float(F.cross_entropy(logits, y, reduction="sum").item())
        correct += int((logits.argmax(dim=-1) == y).sum().item())
        tot += int(y.numel())
    return {"acc": correct / max(1, tot), "nll": nll_sum / max(1, tot)}


def train_cls_with_early_stopping(model, train_loader, val_loader, device):
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=args.LR_CLS, weight_decay=args.WD)

    best_state = None
    best_epoch = 0
    bad = 0

    metric = args.EARLY_STOP_METRIC.lower().strip()
    if metric not in ["nll", "acc"]:
        metric = "nll"

    if metric == "nll":
        best_val = float("inf")
        def is_better(v):
            return (best_val - v) > float(args.EARLY_STOP_MIN_DELTA)
    else:
        best_val = -float("inf")
        def is_better(v):
            return (v - best_val) > float(args.EARLY_STOP_MIN_DELTA)

    for ep in range(1, int(args.EPOCHS_CLS) + 1):
        model.train()
        losses = []
        for Xc, Xn, y in tqdm(train_loader, desc=f"[CLS] ep {ep}/{args.EPOCHS_CLS}", leave=False):
            Xc, Xn, y = Xc.to(device), Xn.to(device), y.to(device)
            _, logits = model(Xc, Xn)
            loss = F.cross_entropy(logits, y)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            losses.append(float(loss.item()))

        m = eval_cls(model, val_loader, device)
        print(f"[CLS] ep={ep:02d} train_loss={np.mean(losses):.4f} val_acc={m['acc']:.4f} val_nll={m['nll']:.4f}")

        cur = float(m[metric])
        if is_better(cur):
            best_val = cur
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_epoch = ep
            bad = 0
        else:
            bad += 1
            if bad >= int(args.EARLY_STOP_PATIENCE):
                print(f"[CLS] Early stopping at ep={ep:02d}. Best ep={best_epoch:02d} best_{metric}={best_val:.6f}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"[CLS] Restored best checkpoint from ep={best_epoch:02d} (best_{metric}={best_val:.6f})")
    return model


def fit_temperature_scaler(model, calib_loader, device) -> TempScaler:
    scaler = TempScaler(init_T=1.0).to(device)
    opt = torch.optim.LBFGS(scaler.parameters(), max_iter=args.TS_MAX_ITERS, line_search_fn="strong_wolfe")

    model.eval()
    logits_list, y_list = [], []
    with torch.no_grad():
        for Xc, Xn, y in calib_loader:
            Xc, Xn, y = Xc.to(device), Xn.to(device), y.to(device)
            _, logits = model(Xc, Xn)
            logits_list.append(logits.detach())
            y_list.append(y)
    logits_all = torch.cat(logits_list, dim=0)
    y_all = torch.cat(y_list, dim=0)

    def closure():
        opt.zero_grad(set_to_none=True)
        scaled = scaler(logits_all)
        loss = F.cross_entropy(scaled, y_all)
        loss.backward()
        return loss

    opt.step(closure)
    with torch.no_grad():
        T = float(torch.exp(scaler.logT).item())
    print(f"[TempScaling] fitted T={T:.4f}")
    return scaler


@torch.no_grad()
def make_feat_batch(model, Xc, Xn, temp_scaler: Optional[TempScaler] = None):
    h, logits_raw = model(Xc, Xn)
    logits = temp_scaler(logits_raw) if temp_scaler is not None else logits_raw
    conf = conf_from_logits(logits)
    ent = entropy_from_logits(logits)
    marg = margin_from_logits(logits)
    feats = torch.cat([h, logits, marg.unsqueeze(-1), ent.unsqueeze(-1)], dim=-1)
    return feats, logits_raw, logits, conf, marg, ent


# -----------------------------
# Stability features
# -----------------------------
class KnowledgeOps:
    def __init__(self, cat_cols: List[str], cont_cols: List[str], X_ref: pd.DataFrame, seed: int):
        self.cat_cols = list(cat_cols)
        self.cont_cols = list(cont_cols)
        self.rng = np.random.default_rng(seed + 2025)

        self.cat_perm_maps: Dict[str, Dict[str, str]] = {}
        for c in self.cat_cols:
            freq = X_ref[c].astype(str).value_counts()
            keys = freq.index[:min(120, len(freq))].tolist()
            perm = keys.copy()
            self.rng.shuffle(perm)
            self.cat_perm_maps[c] = dict(zip(keys, perm))

        self.cont_min, self.cont_max = {}, {}
        for c in self.cont_cols:
            s = X_ref[c].astype(float)
            self.cont_min[c] = float(np.nanpercentile(s, 0.5))
            self.cont_max[c] = float(np.nanpercentile(s, 99.5))

    def perturb(self, X_df: pd.DataFrame, prob: float, cont_std: float) -> pd.DataFrame:
        Xp = X_df.copy()
        n = len(Xp)
        prob = float(np.clip(prob, 0.0, 1.0))

        for c in self.cat_cols:
            mp = self.cat_perm_maps.get(c, {})
            if not mp:
                continue
            s = Xp[c].astype(str)
            in_map = s.isin(mp.keys()).to_numpy()
            flip = self.rng.random(n) < prob
            change = in_map & flip
            if change.any():
                s2 = s.copy()
                s2.loc[change] = s2.loc[change].map(mp).fillna(s2.loc[change])
                Xp[c] = s2.values

        for c in self.cont_cols:
            if n == 0:
                continue
            x = Xp[c].astype(float).to_numpy()
            noise = self.rng.normal(0.0, cont_std, size=n)
            x2 = np.clip(x + noise, self.cont_min[c], self.cont_max[c])
            Xp[c] = x2
        return Xp


@torch.no_grad()
def compute_stability_features(
    model: nn.Module,
    temp_scaler: Optional[TempScaler],
    X_df: pd.DataFrame,
    y: np.ndarray,
    kops: KnowledgeOps,
    cat_cols: List[str],
    cont_cols: List[str],
    cat_maps: Dict[str, Dict[str, int]],
    cont_mean: np.ndarray,
    cont_std: np.ndarray,
    device: torch.device,
    K: int,
    prob: float,
    cont_noise_std: float,
    batch_size: int,
) -> np.ndarray:
    Xc0, Xn0, y0 = encode_split(X_df, y, cat_cols, cont_cols, cat_maps, cont_mean, cont_std)
    loader0 = DataLoader(TabDataset(Xc0, Xn0, y0), batch_size=batch_size, shuffle=False, num_workers=0)

    model.eval()
    if temp_scaler is not None:
        temp_scaler.eval()

    conf0_all, pred0_all = [], []
    for Xc, Xn, _ in loader0:
        Xc, Xn = Xc.to(device), Xn.to(device)
        _, _, logits, conf, _, _ = make_feat_batch(model, Xc, Xn, temp_scaler=temp_scaler)
        conf0_all.append(conf.cpu().numpy())
        pred0_all.append(logits.argmax(dim=-1).cpu().numpy())
    conf0 = np.concatenate(conf0_all).reshape(-1)
    pred0 = np.concatenate(pred0_all).reshape(-1)

    conf_ks, flip_ks, margin_ks, ent_ks = [], [], [], []
    for _ in range(K):
        Xp = kops.perturb(X_df, prob=prob, cont_std=cont_noise_std)
        Xc, Xn, yy = encode_split(Xp, y, cat_cols, cont_cols, cat_maps, cont_mean, cont_std)
        ld = DataLoader(TabDataset(Xc, Xn, yy), batch_size=batch_size, shuffle=False, num_workers=0)

        confk_all, predk_all, marg_all, ent_all = [], [], [], []
        for Xc_b, Xn_b, _ in ld:
            Xc_b, Xn_b = Xc_b.to(device), Xn_b.to(device)
            _, _, logits, confk, margk, entk = make_feat_batch(model, Xc_b, Xn_b, temp_scaler=temp_scaler)
            confk_all.append(confk.cpu().numpy())
            predk_all.append(logits.argmax(dim=-1).cpu().numpy())
            marg_all.append(margk.cpu().numpy())
            ent_all.append(entk.cpu().numpy())

        confk = np.concatenate(confk_all).reshape(-1)
        predk = np.concatenate(predk_all).reshape(-1)
        margk = np.concatenate(marg_all).reshape(-1)
        entk = np.concatenate(ent_all).reshape(-1)

        conf_ks.append(confk)
        flip_ks.append((predk != pred0).astype(np.float32))
        margin_ks.append(margk)
        ent_ks.append(entk)

    conf_ks = np.stack(conf_ks, axis=1)
    flip_ks = np.stack(flip_ks, axis=1)
    margin_ks = np.stack(margin_ks, axis=1)
    ent_ks = np.stack(ent_ks, axis=1)

    conf_drop_max = (conf0 - np.min(conf_ks, axis=1)).astype(np.float32)
    flip_rate = np.mean(flip_ks, axis=1).astype(np.float32)
    margin_std = np.std(margin_ks, axis=1).astype(np.float32)
    ent_mean = np.mean(ent_ks, axis=1).astype(np.float32)
    return np.stack([conf_drop_max, flip_rate, margin_std, ent_mean], axis=1)


# -----------------------------
# Monitoring target + baselines
# -----------------------------
def overconfident_error_label(err: np.ndarray, conf: np.ndarray, tau: float) -> np.ndarray:
    err = np.asarray(err).astype(int).reshape(-1)
    conf = np.asarray(conf).astype(float).reshape(-1)
    return ((err == 1) & (conf >= float(tau))).astype(int)


def fit_red_like_lr(features: np.ndarray, z_target: np.ndarray, seed: int):
    X = features.astype(np.float32)
    y = np.asarray(z_target).astype(int).reshape(-1)
    if len(np.unique(y)) < 2:
        p = float(np.mean(y)) if len(y) else 0.0
        print(f"[RED-like LR] single-class target encountered; using constant predictor p={p:.6f}")
        return ConstantProbModel(p)
    X_tr, X_ev, y_tr, y_ev = _safe_train_test_split(X, y, test_size=0.25, random_state=seed + 17)
    if len(np.unique(y_tr)) < 2:
        print("[RED-like LR] split train fold became single-class; fitting on full data instead of holdout split")
        lr = LogisticRegression(max_iter=2000, class_weight="balanced", n_jobs=1)
        lr.fit(X, y)
        return lr
    lr = LogisticRegression(max_iter=2000, class_weight="balanced", n_jobs=1)
    lr.fit(X_tr, y_tr)
    if len(np.unique(y_ev)) < 2:
        print("[RED-like LR] eval fold is single-class; fitted successfully but skipping AUC print")
        return lr
    print(f"[RED-like LR] AUC={safe_auc(y_ev, lr.predict_proba(X_ev)[:, 1]):.4f}")
    return lr


def _can_stratify_1d(y: np.ndarray, test_size: float) -> bool:
    y = np.asarray(y).reshape(-1)
    if y.size < 2:
        return False
    _, counts = np.unique(y, return_counts=True)
    if len(counts) < 2:
        return False
    if np.min(counts) < 2:
        return False
    n_test = max(1, int(round(float(test_size) * len(y))))
    n_train = len(y) - n_test
    if n_train < len(counts) or n_test < len(counts):
        return False
    return True


def _safe_train_test_split(X, y, test_size: float, random_state: int):
    y = np.asarray(y).reshape(-1)
    stratify = y if _can_stratify_1d(y, test_size) else None
    if stratify is None:
        uniq, counts = np.unique(y, return_counts=True)
        print(
            f"[split] stratify disabled at random_state={random_state} "
            f"(classes={uniq.tolist()}, counts={counts.tolist()}, test_size={test_size})"
        )
    return train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=stratify)


def calibrate_isotonic(scores: np.ndarray, labels: np.ndarray) -> Optional[IsotonicRegression]:
    y, s = _normalize_binary_scores(labels, scores)
    m = np.isfinite(s)
    y = y[m]
    s = s[m]
    if len(y) < 50 or len(np.unique(y)) < 2:
        return None
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(s, y)
    return iso


# -----------------------------
# Detectors: TCP / TCP-M / KOF
# -----------------------------
class TCPHead(nn.Module):
    def __init__(self, in_dim: int, hidden: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


class KOFDetector(nn.Module):
    def __init__(self, in_dim: int, hidden: int, n_tau: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
        )
        self.head = nn.Linear(hidden, n_tau)

    def forward(self, x):
        return self.head(self.net(x))


def make_z_multi_tau(err: torch.Tensor, conf: torch.Tensor, tau_list: List[float]) -> torch.Tensor:
    zs = [(((err > 0.5) & (conf >= float(t))).float()) for t in tau_list]
    return torch.stack(zs, dim=1)


@torch.no_grad()
def kof_head_probs(kof_logits: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-kof_logits))


def fit_tau_combo_lr(head_probs: np.ndarray, z_target: np.ndarray, seed: int):
    X = head_probs.astype(np.float32)
    y = np.asarray(z_target).astype(int).reshape(-1)
    if len(np.unique(y)) < 2:
        p = float(np.mean(y)) if len(y) else 0.0
        print(f"[TauCombo LR] single-class target encountered; using constant predictor p={p:.6f}")
        return ConstantProbModel(p)
    X_tr, X_ev, y_tr, y_ev = _safe_train_test_split(X, y, test_size=0.25, random_state=seed + 101)
    if len(np.unique(y_tr)) < 2:
        print("[TauCombo LR] split train fold became single-class; fitting on full data instead of holdout split")
        lr = LogisticRegression(max_iter=2000, class_weight="balanced", n_jobs=1)
        lr.fit(X, y)
        return lr
    lr = LogisticRegression(max_iter=2000, class_weight="balanced", n_jobs=1)
    lr.fit(X_tr, y_tr)
    if len(np.unique(y_ev)) < 2:
        print("[TauCombo LR] eval fold is single-class; fitted successfully but skipping AUC print")
        return lr
    print(f"[TauCombo LR] AUC={safe_auc(y_ev, lr.predict_proba(X_ev)[:, 1]):.4f}")
    return lr


def train_tcp_correctness(backbone: nn.Module, tcp: nn.Module, feats_loader, device):
    backbone.eval()
    for p in backbone.parameters():
        p.requires_grad = False
    tcp.to(device)
    opt = torch.optim.AdamW(tcp.parameters(), lr=args.LR_DETECT)
    bce = nn.BCEWithLogitsLoss()

    for ep in range(1, int(args.EPOCHS_DETECT) + 1):
        tcp.train()
        losses = []
        for feats, y, pred in tqdm(feats_loader, desc=f"[TCP] ep {ep}/{args.EPOCHS_DETECT}", leave=False):
            feats = feats.to(device)
            y = y.to(device)
            pred = pred.to(device)
            corr = (pred == y).float()
            loss = bce(tcp(feats), corr)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            losses.append(float(loss.item()))
        print(f"[TCP] ep={ep:02d} loss={np.mean(losses):.6f}")


def _sigmoid_np(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-x))


@torch.no_grad()
def _tcpm_val_auc(tcpm: nn.Module, val_feats: torch.Tensor, val_conf: np.ndarray, val_z: np.ndarray, device: torch.device) -> float:
    tcpm.eval()
    logits = tcpm(val_feats.to(device)).detach().cpu().numpy().reshape(-1)
    risk = _sigmoid_np(logits).astype(np.float32)
    if args.TCPM_APPLY_TAU_GATE_AT_INFERENCE:
        risk[val_conf < float(args.TAU_TARGET)] = -1e9
    high = val_conf >= float(args.HIGH_CONF_REGION)
    return safe_auc(val_z[high], risk[high])


def train_tcp_monitoring_strong(
    backbone: nn.Module,
    tcpm: nn.Module,
    feats_train: torch.Tensor,
    z_train: torch.Tensor,
    device: torch.device,
    val_feats: Optional[torch.Tensor] = None,
    val_conf: Optional[np.ndarray] = None,
    val_z: Optional[np.ndarray] = None,
):
    backbone.eval()
    for p in backbone.parameters():
        p.requires_grad = False

    tcpm.to(device)
    opt = torch.optim.AdamW(tcpm.parameters(), lr=args.LR_DETECT)

    if args.TCPM_USE_POS_WEIGHT:
        z_np = z_train.detach().cpu().numpy().astype(np.int64)
        pos = float(z_np.sum())
        neg = float(len(z_np) - pos)
        pw = (neg / max(pos, 1.0)) if pos > 0 else 1.0
        pw = float(np.clip(pw, 1.0, 1e4))
        pos_weight = torch.tensor([pw], dtype=torch.float32, device=device)
        crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        print(f"[TCP-M] pos={int(pos)} neg={int(neg)} pos_weight={pw:.3f} (train slice)")
    else:
        crit = nn.BCEWithLogitsLoss()

    class FeatsForTCPM(Dataset):
        def __init__(self, feats: torch.Tensor, z: torch.Tensor):
            self.feats = feats
            self.z = z
        def __len__(self):
            return int(self.z.numel())
        def __getitem__(self, i):
            return self.feats[i], self.z[i]

    loader = DataLoader(FeatsForTCPM(feats_train, z_train), batch_size=1024, shuffle=True)

    best_state = None
    best_auc = -1.0
    bad = 0

    for ep in range(1, int(args.EPOCHS_TCPM) + 1):
        tcpm.train()
        losses = []
        for feats, z in tqdm(loader, desc=f"[TCP-M] ep {ep}/{args.EPOCHS_TCPM}", leave=False):
            feats = feats.to(device)
            z = z.to(device).float()
            logits = tcpm(feats)
            loss = crit(logits, z)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            losses.append(float(loss.item()))

        auc_val = float("nan")
        if val_feats is not None and val_conf is not None and val_z is not None:
            auc_val = _tcpm_val_auc(tcpm, val_feats, val_conf, val_z, device)
        print(f"[TCP-M] ep={ep:02d} loss={np.mean(losses) if losses else float('nan'):.6f} val_HC_AUC={auc_val:.4f}")

        if np.isfinite(auc_val) and (auc_val - best_auc) > float(args.TCPM_EARLY_STOP_MIN_DELTA):
            best_auc = auc_val
            best_state = {k: v.detach().cpu().clone() for k, v in tcpm.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= int(args.TCPM_EARLY_STOP_PATIENCE):
                print(f"[TCP-M] Early stopping at ep={ep:02d}, best_val_HC_AUC={best_auc:.4f}")
                break

    if best_state is not None:
        tcpm.load_state_dict(best_state)
        print(f"[TCP-M] Restored best checkpoint (val_HC_AUC={best_auc:.4f})")


@torch.no_grad()
def compute_kof_pos_weight(pool_y: torch.Tensor, pool_logits_cal: torch.Tensor) -> np.ndarray:
    conf = conf_from_logits(pool_logits_cal)
    pred = pool_logits_cal.argmax(dim=-1)
    err = (pred != pool_y).float()
    z = make_z_multi_tau(err, conf, args.TAU_LIST).cpu().numpy()
    z_sum = z.sum(axis=0)
    n_sum = z.shape[0]
    pos = np.maximum(z_sum, 1.0)
    neg = np.maximum(n_sum - z_sum, 1.0)
    return (neg / pos).astype(np.float32)


def train_kof_detector(backbone: nn.Module, kof: nn.Module, feats_loader, device, pos_weight_vec: np.ndarray):
    backbone.eval()
    for p in backbone.parameters():
        p.requires_grad = False
    kof.to(device)

    posw = torch.from_numpy(pos_weight_vec).to(device)
    bce = nn.BCEWithLogitsLoss(reduction="none", pos_weight=posw)
    opt = torch.optim.AdamW(kof.parameters(), lr=args.LR_DETECT)

    tau_arr = np.array(args.TAU_LIST, dtype=float)
    ti = int(np.argmin(np.abs(tau_arr - float(args.TAU_TARGET))))
    head_w = np.ones(len(args.TAU_LIST), dtype=np.float32)
    head_w[ti] = 2.5
    head_w_t = torch.from_numpy(head_w).to(device).view(1, -1)

    for ep in range(1, int(args.EPOCHS_DETECT) + 1):
        kof.train()
        losses = []
        for feats, z_multi in tqdm(feats_loader, desc=f"[KOF] ep {ep}/{args.EPOCHS_DETECT}", leave=False):
            feats = feats.to(device)
            z_multi = z_multi.to(device).float()
            qlog = kof(feats)
            lv = bce(qlog, z_multi)
            loss = (lv * head_w_t).mean()
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            losses.append(float(loss.item()))
        print(f"[KOF] ep={ep:02d} loss={np.mean(losses):.6f}")


# -----------------------------
# Fair allocation
# -----------------------------
def _k_from_budget(budget: float, n: int) -> int:
    budget = float(np.clip(budget, 0.0, 1.0))
    return int(np.floor(budget * float(n)))


def _allocate_parity_quotas(g_h: np.ndarray, budget: float) -> Dict[int, int]:
    uniq, counts = np.unique(g_h, return_counts=True)
    n_total = int(counts.sum())
    if n_total <= 0:
        return {}
    K = _k_from_budget(budget, n_total)
    if K <= 0:
        return {int(g): 0 for g in uniq}

    exact = (float(K) * counts.astype(float) / float(n_total))
    base = np.floor(exact).astype(int)
    frac = exact - base
    base = np.clip(base, 0, counts)

    R = K - int(base.sum())
    if R > 0:
        order = np.argsort(-frac)
        for i in order[:R]:
            if base[i] < counts[i]:
                base[i] += 1
    elif R < 0:
        order = np.argsort(frac)
        need = -R
        for i in order:
            if need == 0:
                break
            if base[i] > 0:
                base[i] -= 1
                need -= 1
    return {int(g): int(k) for g, k in zip(uniq, base)}


def _select_alerts_global(r_h: np.ndarray, budget: float) -> np.ndarray:
    n = len(r_h)
    if n == 0:
        return np.zeros(0, dtype=bool)
    K = _k_from_budget(budget, n)
    m = np.zeros(n, dtype=bool)
    if K <= 0:
        return m
    order = np.argsort(-r_h)
    m[order[:K]] = True
    return m


def _select_alerts_parity(r_h: np.ndarray, g_h: np.ndarray, budget: float) -> np.ndarray:
    n = len(r_h)
    if n == 0:
        return np.zeros(0, dtype=bool)
    quotas = _allocate_parity_quotas(g_h, budget)
    m = np.zeros(n, dtype=bool)
    for gg, k in quotas.items():
        idx = np.where(g_h == gg)[0]
        if len(idx) == 0 or k <= 0:
            continue
        order = idx[np.argsort(-r_h[idx])]
        m[order[:k]] = True
    return m


def _select_alerts_bounded_topk(r_h: np.ndarray, g_h: np.ndarray, budget: float, delta: float) -> np.ndarray:
    n = len(r_h)
    if n == 0:
        return np.zeros(0, dtype=bool)

    uniq, counts = np.unique(g_h, return_counts=True)
    n_total = int(counts.sum())
    K = _k_from_budget(budget, n_total)
    if K <= 0:
        return np.zeros(n, dtype=bool)

    delta = float(max(0.0, delta))
    bmin = max(0.0, float(budget) - delta)
    bmax = min(1.0, float(budget) + delta)

    Kmin = np.floor(bmin * counts.astype(float)).astype(int)
    Kmax = np.ceil(bmax * counts.astype(float)).astype(int)
    Kmin = np.clip(Kmin, 0, counts)
    Kmax = np.clip(Kmax, Kmin, counts)

    m = np.zeros(n, dtype=bool)
    picked = {int(g): 0 for g in uniq}

    for gg, kmin in zip(uniq, Kmin):
        gg = int(gg)
        if int(kmin) <= 0:
            continue
        idx = np.where(g_h == gg)[0]
        order = idx[np.argsort(-r_h[idx])]
        take = int(min(int(kmin), len(order)))
        if take > 0:
            m[order[:take]] = True
            picked[gg] += take

    remaining = K - int(m.sum())
    if remaining > 0:
        global_order = np.argsort(-r_h)
        for i in global_order:
            if remaining == 0:
                break
            if m[i]:
                continue
            gg = int(g_h[i])
            gi = int(np.where(uniq == gg)[0][0])
            if picked[gg] < int(Kmax[gi]):
                m[i] = True
                picked[gg] += 1
                remaining -= 1

    if remaining > 0:
        global_order = np.argsort(-r_h)
        for i in global_order:
            if remaining == 0:
                break
            if m[i]:
                continue
            m[i] = True
            remaining -= 1

    return m


def _policy_alerts(r_h: np.ndarray, g_h: np.ndarray, budget: float, policy: str, delta: float) -> np.ndarray:
    if policy == "global":
        return _select_alerts_global(r_h, budget)
    if policy == "parity":
        return _select_alerts_parity(r_h, g_h, budget)
    if policy == "bounded":
        return _select_alerts_bounded_topk(r_h, g_h, budget, delta)
    raise ValueError(f"Unknown policy {policy}")


# -----------------------------
# Evaluation
# -----------------------------
def eval_monitoring_metrics(conf, err, risk_score, group, budgets, high_conf_region, policy="global", delta=0.0, tau_target=0.90, return_alerted=False):
    conf = np.asarray(conf).reshape(-1).astype(float)
    err = np.asarray(err).reshape(-1).astype(int)
    risk = np.asarray(risk_score).reshape(-1).astype(float)
    group = np.asarray(group).reshape(-1).astype(int)

    high = conf >= float(high_conf_region)
    high_tau = conf >= float(tau_target)

    hc_auc_tau = safe_auc(err[high_tau], risk[high_tau]) if high_tau.sum() > 0 else float("nan")
    if high.sum() == 0:
        out = {"hc_auc": float("nan"), "hc_auc_tau": float(hc_auc_tau), "alerts": [], "fairness": [], "groups": []}
        if return_alerted:
            out["alerted_masks"] = {}
        return out

    z = overconfident_error_label(err, conf, tau=float(tau_target))
    z_h, r_h, g_h, conf_h = z[high], risk[high], group[high], conf[high]
    hc_auc = safe_auc(z_h, r_h)

    denom = int(z_h.sum())
    n_high = int(len(z_h))
    z_rate_high = float(denom / n_high) if n_high > 0 else float("nan")

    alerts, fairness, group_rows = [], [], []
    alerted_masks = {}
    delta_out = float(delta) if policy == "bounded" else -1.0

    for b in budgets:
        b = float(b)
        alerted = _policy_alerts(r_h, g_h, b, policy=policy, delta=delta)
        if return_alerted:
            alerted_masks[b] = alerted.copy()

        n_alert = int(alerted.sum())
        tau_mask = conf_h >= float(tau_target)
        n_alert_tau = int((alerted & tau_mask).sum()) if n_alert > 0 else 0
        n_alert_mid = int((alerted & (~tau_mask)).sum()) if n_alert > 0 else 0
        frac_alert_tau = float(n_alert_tau / n_alert) if n_alert > 0 else float("nan")
        frac_alert_mid = float(n_alert_mid / n_alert) if n_alert > 0 else float("nan")

        prec = float(z_h[alerted].mean()) if n_alert > 0 else float("nan")
        rec = float(z_h[alerted].sum() / denom) if denom > 0 else (0.0 if n_alert == 0 else float("nan"))
        lift = float(prec / z_rate_high) if (np.isfinite(prec) and np.isfinite(z_rate_high) and z_rate_high > 0) else float("nan")

        rates, gp, gr, gfpr = [], [], [], []
        n_groups_total = 0
        n_groups_prec_def = 0
        n_groups_rec_def = 0
        n_groups_fpr_def = 0

        for gg in np.unique(g_h):
            m = (g_h == gg)
            n_high_g = int(m.sum())
            if n_high_g == 0:
                continue
            n_groups_total += 1

            n_alert_g = int((alerted & m).sum())
            n_z_g = int(z_h[m].sum())
            n_alert_and_z_g = int(z_h[alerted & m].sum())

            ar = float(n_alert_g / n_high_g) if n_high_g > 0 else float("nan")
            if n_alert_g > 0:
                pg = float(n_alert_and_z_g / n_alert_g)
                n_groups_prec_def += 1
            else:
                pg = float("nan")
            if n_z_g > 0:
                rg = float(n_alert_and_z_g / n_z_g)
                n_groups_rec_def += 1
            else:
                rg = float("nan")

            n_nz_g = n_high_g - n_z_g  # non-failure high-confidence cases in this group
            n_falsealert_g = n_alert_g - n_alert_and_z_g  # alerts raised on non-failures (false alerts)
            if n_nz_g > 0:
                fprg = float(n_falsealert_g / n_nz_g)
                n_groups_fpr_def += 1
            else:
                fprg = float("nan")

            rates.append(ar)
            gp.append(pg)
            gr.append(rg)
            gfpr.append(fprg)

            group_rows.append({
                "budget": b, "policy": policy, "delta": delta_out,
                "group": int(gg),
                "n_high_g": n_high_g, "n_alert_g": n_alert_g, "n_z_g": n_z_g, "n_alert_and_z_g": n_alert_and_z_g,
                "alert_rate_g": ar, "precision_g": pg, "recall_g": rg,
                "n_nonfailure_g": n_nz_g, "n_falsealert_g": n_falsealert_g, "fpr_g": fprg
            })

        if len(rates) >= 2 and np.isfinite(rates).any():
            rmin, rmax = float(np.nanmin(rates)), float(np.nanmax(rates))
            gap_ar = float(rmax - rmin)
            ratio_ar = float(rmax / rmin) if (np.isfinite(rmin) and rmin > 0) else float("nan")
        else:
            gap_ar, ratio_ar = float("nan"), float("nan")

        gap_p = float(np.nanmax(gp) - np.nanmin(gp)) if np.isfinite(gp).any() else float("nan")
        gap_r = float(np.nanmax(gr) - np.nanmin(gr)) if np.isfinite(gr).any() else float("nan")
        gap_fpr = float(np.nanmax(gfpr) - np.nanmin(gfpr)) if np.isfinite(gfpr).any() else float("nan")
        worst_p = float(np.nanmin(gp)) if np.isfinite(gp).any() else float("nan")
        worst_r = float(np.nanmin(gr)) if np.isfinite(gr).any() else float("nan")
        worst_fpr = float(np.nanmax(gfpr)) if np.isfinite(gfpr).any() else float("nan")
        # Equal Opportunity Difference: max gap in TPR (recall) across groups on the high-confidence-failure class
        eq_opportunity_diff = gap_r
        # Equalized Odds Difference: max of TPR gap and FPR gap across groups
        _eo_candidates = [v for v in [gap_r, gap_fpr] if np.isfinite(v)]
        eq_odds_diff = float(np.nanmax(_eo_candidates)) if len(_eo_candidates) else float("nan")

        alerts.append({
            "budget": b, "policy": policy, "delta": delta_out,
            "precision": prec, "recall": rec, "lift": lift,
            "z_rate_high": z_rate_high, "z_count_high": int(denom),
            "worst_precision": worst_p, "worst_recall": worst_r, "worst_fpr": worst_fpr,
            "n_alert": n_alert, "n_high": n_high,
            "n_groups_total": int(n_groups_total),
            "n_groups_precision_defined": int(n_groups_prec_def),
            "n_groups_recall_defined": int(n_groups_rec_def),
            "n_groups_fpr_defined": int(n_groups_fpr_def),
            "n_alert_tau": int(n_alert_tau), "n_alert_mid": int(n_alert_mid),
            "frac_alert_tau": float(frac_alert_tau), "frac_alert_mid": float(frac_alert_mid),
        })

        fairness.append({
            "budget": b, "policy": policy, "delta": delta_out,
            "alert_rate_gap": gap_ar, "alert_rate_ratio": ratio_ar,
            "gap_precision": gap_p, "gap_recall": gap_r, "gap_fpr": gap_fpr,
            "equal_opportunity_diff": eq_opportunity_diff,
            "equalized_odds_diff": eq_odds_diff
        })

    out = {"hc_auc": float(hc_auc), "hc_auc_tau": float(hc_auc_tau), "alerts": alerts, "fairness": fairness, "groups": group_rows}
    if return_alerted:
        out["alerted_masks"] = alerted_masks
    return out


# -----------------------------
# Plot helper
# -----------------------------
def plot_line_ci(df, xcol, mean_col, lo_col, hi_col, hue_col, title, ylabel, outpath, order=None):
    plt.figure()
    groups = order or df[hue_col].unique().tolist()
    for g in groups:
        d = df[df[hue_col] == g].sort_values(xcol)
        if len(d) == 0:
            continue
        x = d[xcol].to_numpy()
        m = d[mean_col].to_numpy()
        lo = d[lo_col].to_numpy()
        hi = d[hi_col].to_numpy()
        plt.plot(x, m, marker="o", label=str(g))
        if np.isfinite(lo).any() and np.isfinite(hi).any():
            plt.fill_between(x, lo, hi, alpha=0.15)
    plt.xlabel(xcol)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(outpath, dpi=160)
    plt.close()


# -----------------------------
# One seed / experiment run
# -----------------------------


def make_safe_validation_splits(d_val: Dict[str, object], seed: int):
    """
    Leakage-safe split of the original validation set into three disjoint parts:
      - val_model  : early stopping only
      - val_select : model / variant selection only
      - val_calib  : temperature scaling + post-hoc calibration / combo fitting only

    Falls back to non-stratified splitting when class counts are too small for
    stratification. This preserves the leakage fix while avoiding crashes on
    tiny validation subsets.
    """
    y = np.asarray(d_val["y"]).reshape(-1).astype(np.int64)
    idx = np.arange(len(y))

    if len(idx) < 6:
        raise RuntimeError("Validation split is too small to create disjoint model/select/calib subsets safely.")

    if len(idx) < 12:
        rng = np.random.default_rng(seed + 2016)
        idx = idx.copy()
        rng.shuffle(idx)
        n_model = max(1, int(round(0.34 * len(idx))))
        n_rest = len(idx) - n_model
        n_select = max(1, n_rest // 2)
        idx_model = idx[:n_model]
        idx_select = idx[n_model:n_model + n_select]
        idx_calib = idx[n_model + n_select:]
        if len(idx_calib) == 0:
            idx_calib = idx_select[-1:]
            idx_select = idx_select[:-1]
    else:
        idx_model, idx_rest, _, y_rest = _safe_train_test_split(
            idx, y, test_size=0.66, random_state=seed + 2017
        )
        if len(idx_rest) < 4:
            rng = np.random.default_rng(seed + 2018)
            idx_rest = np.asarray(idx_rest).copy()
            rng.shuffle(idx_rest)
            half = max(1, len(idx_rest) // 2)
            idx_select = idx_rest[:half]
            idx_calib = idx_rest[half:]
            if len(idx_calib) == 0:
                idx_calib = idx_select[-1:]
                idx_select = idx_select[:-1]
        else:
            idx_select, idx_calib, _, _ = _safe_train_test_split(
                idx_rest, y_rest, test_size=0.50, random_state=seed + 2018
            )

    def _take(ix):
        ix = np.sort(np.asarray(ix))
        return {
            "X": d_val["X"].iloc[ix].reset_index(drop=True),
            "y": np.asarray(d_val["y"])[ix],
            "g": np.asarray(d_val["g"])[ix],
        }

    return _take(idx_model), _take(idx_select), _take(idx_calib)

def run_one_seed_experiment(seed: int, cfg: ExperimentConfig):
    set_seed(seed)
    set_determinism(args.DETERMINISTIC)
    device = get_device(args.DEVICE)
    pin = torch.cuda.is_available()

    print("\n" + "=" * 80)
    print(f"RUN seed={seed} exp={cfg.name} device={device}")
    print("=" * 80)

    splits = load_uci_diabetes_readmission_splits(seed)

    d_train = splits["train"]
    d_val = splits["validation"]
    d_id_test = splits["id_test"]
    d_ood_val = splits["ood_validation"]
    d_ood_test = splits["ood_test"]

    # --- Dataset/shift statistics: base split sizes, group distributions, class imbalance (KBS #2) ---
    dataset_stat_rows: List[Dict[str, object]] = []
    for split_name, dsp in [
        ("train", d_train), ("validation", d_val), ("id_test", d_id_test),
        ("ood_validation", d_ood_val), ("ood_test", d_ood_test)
    ]:
        dataset_stat_rows.append(
            compute_split_stats(f"{cfg.name}/{split_name}", None, dsp["y"], dsp["g"])
        )

    val_model, val_select, val_calib = make_safe_validation_splits(d_val, seed)

    X_train = d_train["X"].copy()
    y_train = d_train["y"].copy()

    swap_col = pick_existing_col(args.SWAP_COL_PREFS, X_train)
    perm_col = pick_existing_col(args.PERM_COL_PREFS, X_train, exclude={swap_col})
    swap_map = build_swap_mapping_label_informed(X_train, y_train, swap_col, args.SWAP_TOPK) if cfg.use_swap_drift else {}
    perm_map = build_perm_mapping_label_free(X_train[perm_col], args.PERM_TOPK, seed + 999) if cfg.use_perm_drift else {}

    cat_cols, cont_cols = infer_cat_cont_cols(X_train, force_cat_cols=[swap_col, perm_col])
    cat_maps, cat_cards, cont_mean, cont_std = fit_encoders(X_train, cat_cols, cont_cols)

    Xc_tr, Xn_tr, y_tr = encode_split(X_train, y_train, cat_cols, cont_cols, cat_maps, cont_mean, cont_std)
    Xc_vm, Xn_vm, y_vm = encode_split(val_model["X"], val_model["y"], cat_cols, cont_cols, cat_maps, cont_mean, cont_std)
    Xc_vs, Xn_vs, y_vs = encode_split(val_select["X"], val_select["y"], cat_cols, cont_cols, cat_maps, cont_mean, cont_std)
    Xc_vc, Xn_vc, y_vc = encode_split(val_calib["X"], val_calib["y"], cat_cols, cont_cols, cat_maps, cont_mean, cont_std)

    train_loader = DataLoader(TabDataset(Xc_tr, Xn_tr, y_tr), batch_size=args.BATCH_SIZE, shuffle=True, num_workers=args.NUM_WORKERS, pin_memory=pin)
    vm_loader = DataLoader(TabDataset(Xc_vm, Xn_vm, y_vm), batch_size=args.BATCH_SIZE, shuffle=False, num_workers=args.NUM_WORKERS, pin_memory=pin)
    vs_loader = DataLoader(TabDataset(Xc_vs, Xn_vs, y_vs), batch_size=args.BATCH_SIZE, shuffle=False, num_workers=args.NUM_WORKERS, pin_memory=pin)
    vc_loader = DataLoader(TabDataset(Xc_vc, Xn_vc, y_vc), batch_size=args.BATCH_SIZE, shuffle=False, num_workers=args.NUM_WORKERS, pin_memory=pin)

    clf = TabMLP(cat_cards, n_cont=len(cont_cols), hidden=args.HIDDEN, dropout=args.DROPOUT).to(device)
    clf = train_cls_with_early_stopping(clf, train_loader, vm_loader, device)
    temp_scaler = fit_temperature_scaler(clf, vc_loader, device) if args.USE_TEMP_SCALING else None

    def extract_monitor_artifacts(loader):
        with torch.no_grad():
            feats_list, conf_list, err_list, ent_list, marg_list = [], [], [], [], []
            for Xc_b, Xn_b, yb in loader:
                Xc_b, Xn_b, yb = Xc_b.to(device), Xn_b.to(device), yb.to(device)
                feats, _, lg_cal, conf, marg, ent = make_feat_batch(clf, Xc_b, Xn_b, temp_scaler=temp_scaler)
                err = (lg_cal.argmax(dim=-1) != yb).long()
                feats_list.append(feats.cpu())
                conf_list.append(conf.cpu())
                err_list.append(err.cpu())
                ent_list.append(ent.cpu())
                marg_list.append(marg.cpu())
        feats = torch.cat(feats_list, dim=0)
        conf = torch.cat(conf_list, dim=0).numpy()
        err = torch.cat(err_list, dim=0).numpy().astype(int)
        ent = torch.cat(ent_list, dim=0).numpy()
        marg = torch.cat(marg_list, dim=0).numpy()
        z = overconfident_error_label(err, conf, args.TAU_TARGET)
        high = conf >= float(args.HIGH_CONF_REGION)
        return feats, conf, err, ent, marg, z, high

    vs_feats, vs_conf, vs_err, vs_ent, vs_marg, z_vs, high_vs = extract_monitor_artifacts(vs_loader)
    vc_feats, vc_conf, vc_err, vc_ent, vc_marg, z_vc, high_vc = extract_monitor_artifacts(vc_loader)

    red_lr = fit_red_like_lr(np.stack([vs_conf, vs_ent, vs_marg], axis=1), z_vs, seed)

    def build_pool():
        """
        Leakage-safe detector pool:
          - base ID-like samples come from val_calib only
          - shifted samples come from ood_validation only
          - mild synthetic ID drift is applied to val_calib only
          - id_test / ood_test are never touched before final evaluation
        """
        rng = np.random.default_rng(seed + 333)

        X_base = val_calib["X"].copy()
        y_base = np.asarray(val_calib["y"]).reshape(-1).astype(np.int64)

        X_shift = d_ood_val["X"].copy()
        y_shift = np.asarray(d_ood_val["y"]).reshape(-1).astype(np.int64)

        if len(y_shift) > 30000:
            idx2 = rng.choice(len(y_shift), size=30000, replace=False)
            idx2 = np.sort(idx2)
            X_shift = X_shift.iloc[idx2].reset_index(drop=True)
            y_shift = y_shift[idx2]

        X_parts = [X_base]
        y_parts = [y_base]

        drift_levels = [0.5, 0.8, 1.0]
        for i, dl in enumerate(drift_levels):
            Xd, _ = apply_combined_drift(
                cfg, X_shift, dl, seed + 700 + i, swap_col, perm_col, swap_map, perm_map, p_override=dl
            )
            X_parts.append(Xd)
            y_parts.append(y_shift)

        X_id_like = val_calib["X"].copy()
        y_id_like = np.asarray(val_calib["y"]).reshape(-1).astype(np.int64)
        if len(y_id_like) > 20000:
            idx3 = rng.choice(len(y_id_like), size=20000, replace=False)
            idx3 = np.sort(idx3)
            X_id_like = X_id_like.iloc[idx3].reset_index(drop=True)
            y_id_like = y_id_like[idx3]
        Xd2, _ = apply_combined_drift(
            cfg, X_id_like, 0.8, seed + 888, swap_col, perm_col, swap_map, perm_map, p_override=0.3
        )
        X_parts.append(Xd2)
        y_parts.append(y_id_like)

        X_all = pd.concat(X_parts, axis=0).reset_index(drop=True)
        y_all = np.concatenate(y_parts).astype(np.int64)
        if len(y_all) > args.STAB_MAX_SAMPLES_FOR_FEAT:
            idxc = rng.choice(len(y_all), size=args.STAB_MAX_SAMPLES_FOR_FEAT, replace=False)
            idxc = np.sort(idxc)
            X_all = X_all.iloc[idxc].reset_index(drop=True)
            y_all = y_all[idxc]
        return X_all, y_all

    X_pool_df, y_pool = build_pool()
    Xc_pool, Xn_pool, y_pool_enc = encode_split(X_pool_df, y_pool, cat_cols, cont_cols, cat_maps, cont_mean, cont_std)
    pool_loader = DataLoader(TabDataset(Xc_pool, Xn_pool, y_pool_enc), batch_size=args.BATCH_SIZE, shuffle=False, num_workers=0)

    kops = KnowledgeOps(cat_cols, cont_cols, X_train, seed)

    stab_pool = None
    stab_vs = None
    stab_vc = None
    if args.USE_STABILITY:
        stab_pool = compute_stability_features(
            clf, temp_scaler, X_pool_df, y_pool, kops, cat_cols, cont_cols, cat_maps, cont_mean, cont_std,
            device, args.STAB_K, args.STAB_PERT_PROB, args.STAB_CONT_NOISE_STD, args.BATCH_SIZE
        )
        stab_vs = compute_stability_features(
            clf, temp_scaler, val_select["X"], val_select["y"], kops, cat_cols, cont_cols, cat_maps, cont_mean, cont_std,
            device, args.STAB_K, args.STAB_PERT_PROB, args.STAB_CONT_NOISE_STD, args.BATCH_SIZE
        )
        stab_vc = compute_stability_features(
            clf, temp_scaler, val_calib["X"], val_calib["y"], kops, cat_cols, cont_cols, cat_maps, cont_mean, cont_std,
            device, args.STAB_K, args.STAB_PERT_PROB, args.STAB_CONT_NOISE_STD, args.BATCH_SIZE
        )

    feats_all, logits_cal_all, conf_all, pred_all = [], [], [], []
    with torch.no_grad():
        for Xc_b, Xn_b, yb in tqdm(pool_loader, desc="[POOL] extracting feats", leave=False):
            Xc_b, Xn_b, yb = Xc_b.to(device), Xn_b.to(device), yb.to(device)
            feats, _, lg_cal, conf, _, _ = make_feat_batch(clf, Xc_b, Xn_b, temp_scaler=temp_scaler)
            pred = lg_cal.argmax(dim=-1)
            feats_all.append(feats.cpu())
            logits_cal_all.append(lg_cal.cpu())
            conf_all.append(conf.cpu())
            pred_all.append(pred.cpu())

    pool_feats = torch.cat(feats_all, dim=0)
    pool_logits_cal = torch.cat(logits_cal_all, dim=0)
    pool_conf = torch.cat(conf_all, dim=0)
    pool_pred = torch.cat(pred_all, dim=0)
    pool_y_t = torch.from_numpy(y_pool_enc).long()

    pool_err = (pool_pred != pool_y_t).long()
    pool_z_tau = ((pool_err > 0) & (pool_conf >= float(args.TAU_TARGET))).long()

    tcp = TCPHead(in_dim=pool_feats.shape[1], hidden=args.KOF_HIDDEN).to(device)

    class _TCPDS(Dataset):
        def __init__(self, feats, y, pred):
            self.feats, self.y, self.pred = feats, y, pred
        def __len__(self):
            return int(self.y.numel())
        def __getitem__(self, i):
            return self.feats[i], self.y[i], self.pred[i]

    train_tcp_correctness(
        clf,
        tcp,
        DataLoader(_TCPDS(pool_feats, pool_y_t, pool_pred), batch_size=1024, shuffle=True),
        device,
    )

    if args.TCPM_TRAIN_ON_TAU_SLICE:
        tcpm_mask = pool_conf >= float(args.TAU_TARGET)
    else:
        tcpm_mask = pool_conf >= float(args.HIGH_CONF_REGION)
    feats_tcpm = pool_feats[tcpm_mask]
    z_tcpm = pool_z_tau[tcpm_mask]

    tcpm = TCPHead(in_dim=pool_feats.shape[1], hidden=args.KOF_HIDDEN).to(device)
    train_tcp_monitoring_strong(
        clf,
        tcpm,
        feats_train=feats_tcpm,
        z_train=z_tcpm,
        device=device,
        val_feats=vs_feats,
        val_conf=vs_conf,
        val_z=z_vs,
    )

    tau_arr = np.array(args.TAU_LIST, dtype=float)
    ti = int(np.argmin(np.abs(tau_arr - float(args.TAU_TARGET))))
    with torch.no_grad():
        z_multi = make_z_multi_tau(pool_err.float(), pool_conf.float(), args.TAU_LIST).cpu()

    kof_nostab = KOFDetector(in_dim=pool_feats.shape[1], hidden=args.KOF_HIDDEN, n_tau=len(args.TAU_LIST)).to(device)
    posw = compute_kof_pos_weight(pool_y_t.to(device), pool_logits_cal.to(device))
    class _KOFDS(Dataset):
        def __init__(self, feats, zmulti):
            self.feats = feats
            self.zm = zmulti
        def __len__(self):
            return int(self.zm.shape[0])
        def __getitem__(self, i):
            return self.feats[i], self.zm[i]

    train_kof_detector(
        clf,
        kof_nostab,
        DataLoader(_KOFDS(pool_feats, z_multi), batch_size=1024, shuffle=True),
        device,
        posw,
    )

    kof_stab = None
    if args.USE_STABILITY and (stab_pool is not None):
        kof_in = torch.cat([pool_feats, torch.from_numpy(stab_pool).float()], dim=1)
        kof_stab = KOFDetector(in_dim=kof_in.shape[1], hidden=args.KOF_HIDDEN, n_tau=len(args.TAU_LIST)).to(device)
        train_kof_detector(
            clf,
            kof_stab,
            DataLoader(_KOFDS(kof_in, z_multi), batch_size=1024, shuffle=True),
            device,
            posw,
        )

    with torch.no_grad():
        lg0_vs = kof_nostab(vs_feats.to(device)).cpu().numpy()
        lg0_vc = kof_nostab(vc_feats.to(device)).cpu().numpy()
        ph0 = kof_head_probs(lg0_vs)
        ph0_vc = kof_head_probs(lg0_vc)
        if kof_stab is not None and stab_vs is not None and stab_vc is not None:
            lgS_vs = kof_stab(torch.cat([vs_feats, torch.from_numpy(stab_vs).float()], dim=1).to(device)).cpu().numpy()
            lgS_vc = kof_stab(torch.cat([vc_feats, torch.from_numpy(stab_vc).float()], dim=1).to(device)).cpu().numpy()
            phS = kof_head_probs(lgS_vs)
            phS_vc = kof_head_probs(lgS_vc)
        else:
            phS = None
            phS_vc = None

    def select_kof_variant(ph_select: np.ndarray, ph_calib: np.ndarray, name: str):
        """
        Leakage-safe KOF post-hoc selection:
          - val_select chooses the variant
          - val_calib fits combo LR and isotonic
        """
        risk_head_select = ph_select[:, ti]
        risk_mean_select = ph_select.mean(axis=1)

        combo_lr = fit_tau_combo_lr(
            ph_calib[high_vc], z_vc[high_vc], seed + (1000 if name == "nostab" else 2000)
        )
        risk_combo_select = combo_lr.predict_proba(ph_select.astype(np.float32))[:, 1]

        best_key, best_auc = None, -1.0
        for k, r in {"head": risk_head_select, "mean": risk_mean_select, "combo": risk_combo_select}.items():
            auc = safe_auc(z_vs[high_vs], r[high_vs])
            print(f"[SelectVariant KOF-{name}] {k}: HC-AUC={auc:.4f}")
            if np.isfinite(auc) and auc > best_auc:
                best_auc, best_key = auc, k
        best_key = best_key or "head"

        risk_head_calib = ph_calib[:, ti]
        risk_mean_calib = ph_calib.mean(axis=1)
        risk_combo_calib = combo_lr.predict_proba(ph_calib.astype(np.float32))[:, 1]
        chosen_calib = {"head": risk_head_calib, "mean": risk_mean_calib, "combo": risk_combo_calib}[best_key]
        iso = calibrate_isotonic(chosen_calib[high_vc], z_vc[high_vc])
        return best_key, combo_lr, iso

    best_nostab, combo_lr_nostab, iso_nostab = select_kof_variant(ph0, ph0_vc, "nostab")
    if phS is not None:
        best_stab, combo_lr_stab, iso_stab = select_kof_variant(phS, phS_vc, "stab")
    else:
        best_stab, combo_lr_stab, iso_stab = None, None, None

    overall_rows, alert_rows, fair_rows, group_rows, selchg_rows = [], [], [], [], []

    X_id, y_id, g_id = d_id_test["X"], d_id_test["y"], d_id_test["g"]
    X_ood, y_ood, g_ood = d_ood_test["X"], d_ood_test["y"], d_ood_test["g"]
    N_TARGET = int(min(args.N_TARGET, len(y_id) + len(y_ood)))

    def sample_target_mix(rho: float, rng: np.random.Generator):
        rho = float(np.clip(rho, 0.0, 1.0))
        n_ood = int(round(N_TARGET * rho))
        n_idd = N_TARGET - n_ood
        n_idd = min(n_idd, len(y_id))
        n_ood = min(n_ood, len(y_ood))

        idx_id = rng.choice(len(y_id), size=n_idd, replace=False) if n_idd > 0 else np.array([], dtype=int)
        idx_oo = rng.choice(len(y_ood), size=n_ood, replace=False) if n_ood > 0 else np.array([], dtype=int)

        Xid = X_id.iloc[idx_id].reset_index(drop=True) if n_idd > 0 else X_id.iloc[:0].copy()
        yid = y_id[idx_id] if n_idd > 0 else np.array([], dtype=np.int64)
        gid = g_id[idx_id] if n_idd > 0 else np.array([], dtype=np.int64)

        Xoo = X_ood.iloc[idx_oo].reset_index(drop=True) if n_ood > 0 else X_ood.iloc[:0].copy()
        yoo = y_ood[idx_oo] if n_ood > 0 else np.array([], dtype=np.int64)
        goo = g_ood[idx_oo] if n_ood > 0 else np.array([], dtype=np.int64)

        Xoo_d, _ = apply_combined_drift(cfg, Xoo, rho, seed + int(1000 * rho) + 101, swap_col, perm_col, swap_map, perm_map)

        if args.DRIFT_ON_ID_AT_HIGH_RHO:
            p_id = float(np.clip(args.ID_DRIFT_SCALE * rho, 0.0, 1.0))
            Xid_d, _ = apply_combined_drift(cfg, Xid, rho, seed + int(1000 * rho) + 202, swap_col, perm_col, swap_map, perm_map, p_override=p_id)
        else:
            Xid_d = Xid

        X_all = pd.concat([Xid_d, Xoo_d], axis=0).reset_index(drop=True)
        y_all = np.concatenate([yid, yoo]).astype(np.int64)
        g_all = np.concatenate([gid, goo]).astype(np.int64)

        perm = rng.permutation(len(y_all))
        return X_all.iloc[perm].reset_index(drop=True), y_all[perm], g_all[perm]

    def score_batch(X_df: pd.DataFrame, y: np.ndarray):
        Xc_te, Xn_te, y_te = encode_split(X_df, y, cat_cols, cont_cols, cat_maps, cont_mean, cont_std)
        te_loader = DataLoader(TabDataset(Xc_te, Xn_te, y_te), batch_size=args.BATCH_SIZE, shuffle=False, num_workers=0)

        feats, logits_cal, conf, err, ent, marg, ener, pred = [], [], [], [], [], [], [], []
        with torch.no_grad():
            for Xc_b, Xn_b, yb in te_loader:
                Xc_b, Xn_b, yb = Xc_b.to(device), Xn_b.to(device), yb.to(device)
                f, _, lg_cal_b, cf, mg, en = make_feat_batch(clf, Xc_b, Xn_b, temp_scaler=temp_scaler)
                er = (lg_cal_b.argmax(dim=-1) != yb).long()
                feats.append(f.cpu())
                logits_cal.append(lg_cal_b.cpu())
                conf.append(cf.cpu())
                err.append(er.cpu())
                ent.append(en.cpu())
                marg.append(mg.cpu())
                ener.append(energy_from_logits(lg_cal_b).cpu())
                pred.append(lg_cal_b.argmax(dim=-1).cpu())
        feats = torch.cat(feats, dim=0)
        logits_cal = torch.cat(logits_cal, dim=0)
        conf = torch.cat(conf, dim=0).numpy()
        err = torch.cat(err, dim=0).numpy().astype(int)
        ent = torch.cat(ent, dim=0).numpy()
        marg = torch.cat(marg, dim=0).numpy()
        ener = torch.cat(ener, dim=0).numpy()
        pred = torch.cat(pred, dim=0).numpy().astype(int)
        return feats, logits_cal, conf, err, pred, ent, marg, ener

    for rho in args.RHO_VALUES:
        rng_e = np.random.default_rng(seed + 5000 + int(1e6 * rho))
        X_te_df, y_te, g_te = sample_target_mix(rho, rng_e)
        feats_te, logits_cal_te, conf_te, err_te, pred_te, ent_te, marg_te, energy_te = score_batch(X_te_df, y_te)

        # --- Dataset/shift statistics under this shift condition (KBS #2) ---
        z_te_stats = overconfident_error_label(err_te, conf_te, args.TAU_TARGET)
        high_mask_te_stats = conf_te >= float(args.HIGH_CONF_REGION)
        dataset_stat_rows.append(
            compute_split_stats(f"{cfg.name}/target_mix", float(rho), y_te, g_te,
                                 z=z_te_stats, high_mask=high_mask_te_stats)
        )

        stab_te = None
        if args.USE_STABILITY and (kof_stab is not None):
            stab_te = compute_stability_features(
                clf, temp_scaler, X_te_df, y_te, kops, cat_cols, cont_cols, cat_maps, cont_mean, cont_std,
                device, args.STAB_K, args.STAB_PERT_PROB, args.STAB_CONT_NOISE_STD, args.BATCH_SIZE
            )

        risk_ent = ent_te
        risk_energy = energy_te
        risk_red = red_lr.predict_proba(np.stack([conf_te, ent_te, marg_te], axis=1).astype(np.float32))[:, 1]

        with torch.no_grad():
            tcp_logit = tcp(feats_te.to(device)).cpu().numpy().reshape(-1)
        risk_tcp = 1.0 - _sigmoid_np(tcp_logit)

        risk_tcp_gate_tau = risk_tcp.copy()
        risk_tcp_gate_tau[conf_te < float(args.TAU_TARGET)] = -1e9

        rng_rand = np.random.default_rng(seed + 90000 + int(1e6 * float(rho)))
        risk_rand_gate_tau = rng_rand.random(len(conf_te)).astype(np.float32)
        risk_rand_gate_tau[conf_te < float(args.TAU_TARGET)] = -1e9

        with torch.no_grad():
            tcpm_logit = tcpm(feats_te.to(device)).cpu().numpy().reshape(-1)
        risk_tcpm = _sigmoid_np(tcpm_logit).astype(np.float32)
        risk_tcpm_gate_tau = risk_tcpm.copy()
        risk_tcpm_gate_tau[conf_te < float(args.TAU_TARGET)] = -1e9

        with torch.no_grad():
            lg0 = kof_nostab(feats_te.to(device)).cpu().numpy()
            ph0 = kof_head_probs(lg0)
        risk_kof_nostab = {
            "head": ph0[:, ti],
            "mean": ph0.mean(axis=1),
            "combo": combo_lr_nostab.predict_proba(ph0.astype(np.float32))[:, 1]
        }[best_nostab]
        if iso_nostab is not None:
            high_te = conf_te >= float(args.HIGH_CONF_REGION)
            r = risk_kof_nostab.copy()
            r[high_te] = iso_nostab.predict(r[high_te])
            risk_kof_nostab = r

        if kof_stab is not None and stab_te is not None and combo_lr_stab is not None:
            with torch.no_grad():
                lgS = kof_stab(torch.cat([feats_te, torch.from_numpy(stab_te).float()], dim=1).to(device)).cpu().numpy()
                phS = kof_head_probs(lgS)
            risk_kof_stab = {
                "head": phS[:, ti],
                "mean": phS.mean(axis=1),
                "combo": combo_lr_stab.predict_proba(phS.astype(np.float32))[:, 1]
            }[best_stab]
            if iso_stab is not None:
                high_te = conf_te >= float(args.HIGH_CONF_REGION)
                r = risk_kof_stab.copy()
                r[high_te] = iso_stab.predict(r[high_te])
                risk_kof_stab = r
        else:
            risk_kof_stab = risk_kof_nostab

        risk_kof_stab_gate_tau = risk_kof_stab.copy()
        risk_kof_stab_gate_tau[conf_te < float(args.TAU_TARGET)] = -1e9

        res_kof_nostab = eval_monitoring_metrics(conf_te, err_te, risk_kof_nostab, g_te, args.ALERT_BUDGETS, args.HIGH_CONF_REGION, policy="global", delta=0.0, tau_target=args.TAU_TARGET, return_alerted=True)
        res_kof_stab = eval_monitoring_metrics(conf_te, err_te, risk_kof_stab, g_te, args.ALERT_BUDGETS, args.HIGH_CONF_REGION, policy="global", delta=0.0, tau_target=args.TAU_TARGET, return_alerted=True)
        res_kof_stab_gate = eval_monitoring_metrics(conf_te, err_te, risk_kof_stab_gate_tau, g_te, args.ALERT_BUDGETS, args.HIGH_CONF_REGION, policy="global", delta=0.0, tau_target=args.TAU_TARGET, return_alerted=True)
        res_tcp = eval_monitoring_metrics(conf_te, err_te, risk_tcp, g_te, args.ALERT_BUDGETS, args.HIGH_CONF_REGION, policy="global", delta=0.0, tau_target=args.TAU_TARGET, return_alerted=True)
        res_tcp_gate = eval_monitoring_metrics(conf_te, err_te, risk_tcp_gate_tau, g_te, args.ALERT_BUDGETS, args.HIGH_CONF_REGION, policy="global", delta=0.0, tau_target=args.TAU_TARGET, return_alerted=True)
        res_rand_gate = eval_monitoring_metrics(conf_te, err_te, risk_rand_gate_tau, g_te, args.ALERT_BUDGETS, args.HIGH_CONF_REGION, policy="global", delta=0.0, tau_target=args.TAU_TARGET, return_alerted=False)
        res_tcpm = eval_monitoring_metrics(conf_te, err_te, risk_tcpm, g_te, args.ALERT_BUDGETS, args.HIGH_CONF_REGION, policy="global", delta=0.0, tau_target=args.TAU_TARGET, return_alerted=True)
        res_tcpm_gate = eval_monitoring_metrics(conf_te, err_te, risk_tcpm_gate_tau, g_te, args.ALERT_BUDGETS, args.HIGH_CONF_REGION, policy="global", delta=0.0, tau_target=args.TAU_TARGET, return_alerted=True)
        res_red = eval_monitoring_metrics(conf_te, err_te, risk_red, g_te, args.ALERT_BUDGETS, args.HIGH_CONF_REGION, policy="global", delta=0.0, tau_target=args.TAU_TARGET, return_alerted=False)
        res_ent = eval_monitoring_metrics(conf_te, err_te, risk_ent, g_te, args.ALERT_BUDGETS, args.HIGH_CONF_REGION, policy="global", delta=0.0, tau_target=args.TAU_TARGET, return_alerted=False)
        res_energy = eval_monitoring_metrics(conf_te, err_te, risk_energy, g_te, args.ALERT_BUDGETS, args.HIGH_CONF_REGION, policy="global", delta=0.0, tau_target=args.TAU_TARGET, return_alerted=False)

        mitig = []
        if args.ENABLE_FAIRNESS_MITIGATION:
            base_masks = {
                "tcp_gate_tau": res_tcp_gate.get("alerted_masks", {}),
                "tcp_m_gate_tau": res_tcpm_gate.get("alerted_masks", {}),
                "kof_stab": res_kof_stab.get("alerted_masks", {}),
                "kof_stab_gate_tau": res_kof_stab_gate.get("alerted_masks", {}),
            }
            risk_map = {
                "tcp_gate_tau": risk_tcp_gate_tau,
                "tcp_m_gate_tau": risk_tcpm_gate_tau,
                "kof_stab": risk_kof_stab,
                "kof_stab_gate_tau": risk_kof_stab_gate_tau,
            }

            for mname in args.MITIGATE_MODELS:
                if mname not in risk_map:
                    continue
                rv = risk_map[mname]
                bm = base_masks.get(mname, {})

                if "parity" in args.FAIRNESS_POLICIES:
                    rr = eval_monitoring_metrics(conf_te, err_te, rv, g_te, args.ALERT_BUDGETS, args.HIGH_CONF_REGION, policy="parity", delta=0.0, tau_target=args.TAU_TARGET, return_alerted=True)
                    mitig.append((mname, rr))
                    for b in args.ALERT_BUDGETS:
                        b = float(b)
                        if (b in rr.get("alerted_masks", {})) and (b in bm):
                            selchg_rows.append({
                                "exp": cfg.name, "seed": seed, "rho": float(rho), "model": mname, "policy": "parity", "delta": -1.0, "budget": b,
                                "n_selection_changed": int(np.sum(rr["alerted_masks"][b] != bm[b]))
                            })

                if "bounded" in args.FAIRNESS_POLICIES:
                    for dd in args.FAIR_TOPK_DELTAS:
                        rr = eval_monitoring_metrics(conf_te, err_te, rv, g_te, args.ALERT_BUDGETS, args.HIGH_CONF_REGION, policy="bounded", delta=float(dd), tau_target=args.TAU_TARGET, return_alerted=True)
                        mitig.append((mname, rr))
                        for b in args.ALERT_BUDGETS:
                            b = float(b)
                            if (b in rr.get("alerted_masks", {})) and (b in bm):
                                selchg_rows.append({
                                    "exp": cfg.name, "seed": seed, "rho": float(rho), "model": mname, "policy": "bounded", "delta": float(dd), "budget": b,
                                    "n_selection_changed": int(np.sum(rr["alerted_masks"][b] != bm[b]))
                                })

        def _append(model_name: str, res):
            for it in res["alerts"]:
                alert_rows.append({"exp": cfg.name, "seed": seed, "rho": float(rho), "model": model_name, **it})
            for it in res["fairness"]:
                fair_rows.append({"exp": cfg.name, "seed": seed, "rho": float(rho), "model": model_name, **it})
            for it in res["groups"]:
                group_rows.append({"exp": cfg.name, "seed": seed, "rho": float(rho), "model": model_name, **it})

        _append("kof_nostab", res_kof_nostab)
        _append("kof_stab", res_kof_stab)
        _append("kof_stab_gate_tau", res_kof_stab_gate)
        _append("tcp", res_tcp)
        _append("tcp_gate_tau", res_tcp_gate)
        _append("rand_gate_tau", res_rand_gate)
        _append("tcp_m", res_tcpm)
        _append("tcp_m_gate_tau", res_tcpm_gate)
        _append("red_lr", res_red)
        _append("entropy", res_ent)
        _append("energy", res_energy)
        for mname, rr in mitig:
            _append(mname, rr)

        overall_rows.append({
            "exp": cfg.name, "seed": seed, "rho": float(rho),
            "n_test": int(len(err_te)),
            "high_conf_n": int((conf_te >= args.HIGH_CONF_REGION).sum()),
            "hc_auc_kof_nostab": float(res_kof_nostab["hc_auc"]),
            "hc_auc_tau_kof_nostab": float(res_kof_nostab["hc_auc_tau"]),
            "hc_auc_kof_stab": float(res_kof_stab["hc_auc"]),
            "hc_auc_tau_kof_stab": float(res_kof_stab["hc_auc_tau"]),
            "hc_auc_kof_stab_gate_tau": float(res_kof_stab_gate["hc_auc"]),
            "hc_auc_tau_kof_stab_gate_tau": float(res_kof_stab_gate["hc_auc_tau"]),
            "hc_auc_tcp": float(res_tcp["hc_auc"]),
            "hc_auc_tau_tcp": float(res_tcp["hc_auc_tau"]),
            "hc_auc_tcp_gate_tau": float(res_tcp_gate["hc_auc"]),
            "hc_auc_tau_tcp_gate_tau": float(res_tcp_gate["hc_auc_tau"]),
            "hc_auc_rand_gate_tau": float(res_rand_gate["hc_auc"]),
            "hc_auc_tau_rand_gate_tau": float(res_rand_gate["hc_auc_tau"]),
            "hc_auc_tcp_m": float(res_tcpm["hc_auc"]),
            "hc_auc_tau_tcp_m": float(res_tcpm["hc_auc_tau"]),
            "hc_auc_tcp_m_gate_tau": float(res_tcpm_gate["hc_auc"]),
            "hc_auc_tau_tcp_m_gate_tau": float(res_tcpm_gate["hc_auc_tau"]),
            "hc_auc_red": float(res_red["hc_auc"]),
            "hc_auc_tau_red": float(res_red["hc_auc_tau"]),
            "hc_auc_entropy": float(res_ent["hc_auc"]),
            "hc_auc_tau_entropy": float(res_ent["hc_auc_tau"]),
            "hc_auc_energy": float(res_energy["hc_auc"]),
            "hc_auc_tau_energy": float(res_energy["hc_auc_tau"]),
            "best_variant_nostab": best_nostab,
            "best_variant_stab": best_stab if best_stab is not None else "NA",
            "isotonic_nostab": int(iso_nostab is not None),
            "isotonic_stab": int(iso_stab is not None),
        })

        print(
            f"[rho={rho:.2f}] HC-AUC: TCP={res_tcp['hc_auc']:.4f} "
            f"TCP-M={res_tcpm['hc_auc']:.4f} TCP-M(gate)={res_tcpm_gate['hc_auc']:.4f} "
            f"KOF(stab)={res_kof_stab['hc_auc']:.4f}"
        )

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    meta = {
        "seed": seed,
        "exp": cfg.name,
        "dataset": DATASET_NAME,
        "uci_id": 296,
        "ood_definition": "admission_source_id == 7",
        "sensitive_col_eval_only": SENSITIVE_COL,
        "gender_keep": list(GENDER_KEEP),
        "swap_col": swap_col,
        "perm_col": perm_col,
        "swap_map_size": len(swap_map),
        "perm_map_size": len(perm_map),
        "tau_target": args.TAU_TARGET,
        "tau_list": args.TAU_LIST,
        "high_conf_region": args.HIGH_CONF_REGION,
        "alert_budgets": args.ALERT_BUDGETS,
        "fairness_mitigation_enabled": int(args.ENABLE_FAIRNESS_MITIGATION),
        "fairness_policies": args.FAIRNESS_POLICIES,
        "fair_topk_deltas": args.FAIR_TOPK_DELTAS,
        "mitigate_models": args.MITIGATE_MODELS,
        "group_min_n_list": args.GROUP_MIN_N_LIST,
        "bootstrap_n": args.BOOTSTRAP_N,
        "bootstrap_alpha": args.BOOTSTRAP_ALPHA,
    }

    return (
        pd.DataFrame(overall_rows),
        pd.DataFrame(alert_rows),
        pd.DataFrame(fair_rows),
        pd.DataFrame(group_rows),
        pd.DataFrame(selchg_rows),
        meta,
        pd.DataFrame(dataset_stat_rows),
    )


# -----------------------------
# Main
# -----------------------------
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--outdir", type=str, default=args.OUTDIR)
    parser.add_argument("--data_dir", type=str, default=args.DATA_DIR)
    parser.add_argument("--device", type=str, default=args.DEVICE)
    parser.add_argument("--max_train", type=int, default=args.MAX_TRAIN)
    parser.add_argument("--max_val", type=int, default=args.MAX_VAL)
    parser.add_argument("--n_target", type=int, default=args.N_TARGET)
    parser.add_argument("--batch_size", type=int, default=args.BATCH_SIZE)
    parser.add_argument("--epochs_cls", type=int, default=args.EPOCHS_CLS)
    parser.add_argument("--epochs_detect", type=int, default=args.EPOCHS_DETECT)
    parser.add_argument("--epochs_tcpm", type=int, default=args.EPOCHS_TCPM)
    parser.add_argument("--early_stop_metric", type=str, default=args.EARLY_STOP_METRIC, choices=["nll", "acc"])
    parser.add_argument("--early_stop_patience", type=int, default=args.EARLY_STOP_PATIENCE)
    parser.add_argument("--early_stop_min_delta", type=float, default=args.EARLY_STOP_MIN_DELTA)
    parser.add_argument("--tcpm_patience", type=int, default=args.TCPM_EARLY_STOP_PATIENCE)
    parser.add_argument("--tcpm_min_delta", type=float, default=args.TCPM_EARLY_STOP_MIN_DELTA)
    opt, _ = parser.parse_known_args()

    args.OUTDIR = opt.outdir
    args.DATA_DIR = opt.data_dir
    args.DEVICE = opt.device
    args.MAX_TRAIN = opt.max_train
    args.MAX_VAL = opt.max_val
    args.N_TARGET = int(opt.n_target)
    args.BATCH_SIZE = opt.batch_size
    args.EPOCHS_CLS = opt.epochs_cls
    args.EPOCHS_DETECT = opt.epochs_detect
    args.EPOCHS_TCPM = opt.epochs_tcpm
    args.EARLY_STOP_METRIC = opt.early_stop_metric
    args.EARLY_STOP_PATIENCE = opt.early_stop_patience
    args.EARLY_STOP_MIN_DELTA = opt.early_stop_min_delta
    args.TCPM_EARLY_STOP_PATIENCE = opt.tcpm_patience
    args.TCPM_EARLY_STOP_MIN_DELTA = opt.tcpm_min_delta

    ensure_outdirs(args.OUTDIR)
    TABLES = os.path.join(args.OUTDIR, "tables")
    FIGS = os.path.join(args.OUTDIR, "figs")

    versions = dump_versions()
    print("VERSIONS:", versions)
    print("CONFIG:", json.dumps(asdict(args), indent=2))
    print("DATASET:", DATASET_NAME, "(UCI id=296; OOD admission_source_id == 7)", "Sensitive(eval-only):", SENSITIVE_COL, "Gender keep:", GENDER_KEEP)

    ALL_OVERALL, ALL_ALERTS, ALL_FAIR, ALL_GROUP, ALL_SELCHG, ALL_META, ALL_DSTATS = [], [], [], [], [], [], []

    for cfg in experiments():
        for sd in args.SEEDS:
            o, a, fdf, gdf, selchg, meta, dstats = run_one_seed_experiment(sd, cfg)
            ALL_OVERALL.append(o)
            ALL_ALERTS.append(a)
            ALL_FAIR.append(fdf)
            ALL_GROUP.append(gdf)
            if len(selchg):
                ALL_SELCHG.append(selchg)
            ALL_META.append(meta)
            ALL_DSTATS.append(dstats.assign(seed=sd))

    raw_overall = pd.concat(ALL_OVERALL, axis=0).reset_index(drop=True)
    raw_alerts = pd.concat(ALL_ALERTS, axis=0).reset_index(drop=True)
    raw_fair = pd.concat(ALL_FAIR, axis=0).reset_index(drop=True)
    raw_group = pd.concat(ALL_GROUP, axis=0).reset_index(drop=True)
    raw_selchg = pd.concat(ALL_SELCHG, axis=0).reset_index(drop=True) if len(ALL_SELCHG) else pd.DataFrame()
    raw_meta = pd.DataFrame(ALL_META)

    raw_overall.to_csv(os.path.join(TABLES, "raw_monitor_overall.csv"), index=False)
    raw_alerts.to_csv(os.path.join(TABLES, "raw_monitor_alerts.csv"), index=False)
    raw_fair.to_csv(os.path.join(TABLES, "raw_monitor_fairness_alerts.csv"), index=False)
    raw_group.to_csv(os.path.join(TABLES, "raw_monitor_group_alerts.csv"), index=False)
    if len(raw_selchg):
        raw_selchg.to_csv(os.path.join(TABLES, "raw_mitigation_selection_changes.csv"), index=False)
    raw_meta.to_csv(os.path.join(TABLES, "run_metadata.csv"), index=False)

    raw_dataset_stats = pd.concat(ALL_DSTATS, axis=0).reset_index(drop=True) if len(ALL_DSTATS) else pd.DataFrame()
    if len(raw_dataset_stats):
        raw_dataset_stats.to_csv(os.path.join(TABLES, "raw_dataset_shift_statistics.csv"), index=False)
        dstat_metric_cols = [c for c in raw_dataset_stats.columns if c not in ("split", "rho", "seed")]
        dataset_stats_summary = mean_std_ci_over_seeds(
            raw_dataset_stats,
            ["split", "rho"],
            dstat_metric_cols,
            seed_col="seed", n_boot=args.BOOTSTRAP_N, alpha=args.BOOTSTRAP_ALPHA
        )
        dataset_stats_summary.to_csv(os.path.join(TABLES, "summary_dataset_shift_statistics_mean_std_ci.csv"), index=False)
    else:
        dataset_stats_summary = pd.DataFrame()

    overall_summary = mean_std_ci_over_seeds(
        raw_overall,
        ["exp", "rho"],
        [
            "hc_auc_kof_nostab", "hc_auc_tau_kof_nostab",
            "hc_auc_kof_stab", "hc_auc_tau_kof_stab",
            "hc_auc_kof_stab_gate_tau", "hc_auc_tau_kof_stab_gate_tau",
            "hc_auc_tcp", "hc_auc_tau_tcp",
            "hc_auc_tcp_gate_tau", "hc_auc_tau_tcp_gate_tau",
            "hc_auc_rand_gate_tau", "hc_auc_tau_rand_gate_tau",
            "hc_auc_tcp_m", "hc_auc_tau_tcp_m",
            "hc_auc_tcp_m_gate_tau", "hc_auc_tau_tcp_m_gate_tau",
            "hc_auc_red", "hc_auc_tau_red",
            "hc_auc_entropy", "hc_auc_tau_entropy",
            "hc_auc_energy", "hc_auc_tau_energy",
            "high_conf_n"
        ],
        seed_col="seed", n_boot=args.BOOTSTRAP_N, alpha=args.BOOTSTRAP_ALPHA
    )
    overall_summary.to_csv(os.path.join(TABLES, "summary_monitor_overall_mean_std_ci.csv"), index=False)

    alerts_summary = mean_std_ci_over_seeds(
        raw_alerts,
        ["exp", "rho", "model", "policy", "delta", "budget"],
        [
            "precision", "recall", "lift", "z_rate_high",
            "worst_precision", "worst_recall",
            "n_groups_total", "n_alert", "n_high",
            "n_alert_tau", "n_alert_mid",
            "frac_alert_tau", "frac_alert_mid"
        ],
        seed_col="seed", n_boot=args.BOOTSTRAP_N, alpha=args.BOOTSTRAP_ALPHA
    )
    alerts_summary.to_csv(os.path.join(TABLES, "summary_monitor_alerts_mean_std_ci.csv"), index=False)

    fair_summary = mean_std_ci_over_seeds(
        raw_fair,
        ["exp", "rho", "model", "policy", "delta", "budget"],
        ["alert_rate_gap", "alert_rate_ratio", "gap_precision", "gap_recall", "gap_fpr",
         "equal_opportunity_diff", "equalized_odds_diff"],
        seed_col="seed", n_boot=args.BOOTSTRAP_N, alpha=args.BOOTSTRAP_ALPHA
    )
    fair_summary.to_csv(os.path.join(TABLES, "summary_monitor_fairness_mean_std_ci.csv"), index=False)

    group_summary = mean_std_ci_over_seeds(
        raw_group,
        ["exp", "rho", "model", "policy", "delta", "budget", "group"],
        ["n_high_g", "n_alert_g", "n_z_g", "n_alert_and_z_g", "alert_rate_g", "precision_g", "recall_g",
         "n_nonfailure_g", "n_falsealert_g", "fpr_g"],
        seed_col="seed", n_boot=args.BOOTSTRAP_N, alpha=args.BOOTSTRAP_ALPHA
    )
    group_summary.to_csv(os.path.join(TABLES, "summary_monitor_group_mean_std_ci.csv"), index=False)

    for MIN_N in args.GROUP_MIN_N_LIST:
        MIN_N = int(MIN_N)
        if MIN_N <= 1:
            continue
        filt = raw_group[raw_group["n_high_g"] >= MIN_N].copy()
        gs = mean_std_ci_over_seeds(
            filt,
            ["exp", "rho", "model", "policy", "delta", "budget", "group"],
            ["n_high_g", "n_alert_g", "n_z_g", "n_alert_and_z_g", "alert_rate_g", "precision_g", "recall_g",
             "n_nonfailure_g", "n_falsealert_g", "fpr_g"],
            seed_col="seed", n_boot=args.BOOTSTRAP_N, alpha=args.BOOTSTRAP_ALPHA
        )
        gs.to_csv(os.path.join(TABLES, f"summary_monitor_group_mean_std_ci_minN{MIN_N}.csv"), index=False)

    tradeoff_raw = raw_alerts.merge(raw_fair, on=["exp", "seed", "rho", "model", "policy", "delta", "budget"], how="inner")
    tradeoff_raw.to_csv(os.path.join(TABLES, "raw_monitor_tradeoff.csv"), index=False)
    tradeoff_summary = mean_std_ci_over_seeds(
        tradeoff_raw,
        ["exp", "rho", "model", "policy", "delta", "budget"],
        ["precision", "recall", "alert_rate_gap", "worst_precision", "worst_recall", "worst_fpr",
         "lift", "z_rate_high", "gap_fpr", "equal_opportunity_diff", "equalized_odds_diff"],
        seed_col="seed", n_boot=args.BOOTSTRAP_N, alpha=args.BOOTSTRAP_ALPHA
    )
    tradeoff_summary.to_csv(os.path.join(TABLES, "summary_monitor_tradeoff_mean_std_ci.csv"), index=False)

    BSEL = float(args.ALERT_BUDGETS[0])

    full_over = overall_summary[overall_summary["exp"] == "full"].copy()
    if len(full_over):
        rows = []
        for _, r in full_over.iterrows():
            for m, col in [
                ("tcp", "hc_auc_tcp"),
                ("tcp_m", "hc_auc_tcp_m"),
                ("tcp_m_gate_tau", "hc_auc_tcp_m_gate_tau"),
                ("kof_stab", "hc_auc_kof_stab"),
                ("kof_stab_gate_tau", "hc_auc_kof_stab_gate_tau"),
                ("kof_nostab", "hc_auc_kof_nostab"),
                ("red_lr", "hc_auc_red"),
            ]:
                rows.append({
                    "rho": r["rho"], "model": m,
                    "mean": r[f"{col}_mean"], "lo": r[f"{col}_ci_low"], "hi": r[f"{col}_ci_high"]
                })
        dfp = pd.DataFrame(rows)
        plot_line_ci(
            dfp, "rho", "mean", "lo", "hi", "model",
            "HC-AUC vs rho (FULL, mean + bootstrap CI)", "HC-AUC",
            os.path.join(FIGS, "hc_auc_vs_rho.png"),
            order=["tcp", "tcp_m", "tcp_m_gate_tau", "kof_stab", "kof_stab_gate_tau", "kof_nostab", "red_lr"]
        )

    subA = alerts_summary[
        (alerts_summary["exp"] == "full") &
        (np.isclose(alerts_summary["budget"], BSEL)) &
        (alerts_summary["policy"] == "global") &
        (alerts_summary["delta"] == -1.0)
    ].copy()
    if len(subA):
        p_df = subA[["rho", "model", "precision_mean", "precision_ci_low", "precision_ci_high"]].rename(
            columns={"precision_mean": "mean", "precision_ci_low": "lo", "precision_ci_high": "hi"}
        )
        plot_line_ci(
            p_df, "rho", "mean", "lo", "hi", "model",
            f"Alert precision vs rho @ budget={BSEL:.2f} (FULL, global)", "precision",
            os.path.join(FIGS, "alert_precision_vs_rho.png"),
            order=["tcp", "tcp_m", "tcp_m_gate_tau", "kof_stab", "kof_stab_gate_tau", "kof_nostab", "red_lr", "entropy", "energy"]
        )

        r_df = subA[["rho", "model", "recall_mean", "recall_ci_low", "recall_ci_high"]].rename(
            columns={"recall_mean": "mean", "recall_ci_low": "lo", "recall_ci_high": "hi"}
        )
        plot_line_ci(
            r_df, "rho", "mean", "lo", "hi", "model",
            f"Alert recall vs rho @ budget={BSEL:.2f} (FULL, global)", "recall",
            os.path.join(FIGS, "alert_recall_vs_rho.png"),
            order=["tcp", "tcp_m", "tcp_m_gate_tau", "kof_stab", "kof_stab_gate_tau", "kof_nostab", "red_lr", "entropy", "energy"]
        )

        l_df = subA[["rho", "model", "lift_mean", "lift_ci_low", "lift_ci_high"]].rename(
            columns={"lift_mean": "mean", "lift_ci_low": "lo", "lift_ci_high": "hi"}
        )
        plot_line_ci(
            l_df, "rho", "mean", "lo", "hi", "model",
            f"Alert lift vs rho @ budget={BSEL:.2f} (FULL, global)", "lift (precision/base-rate)",
            os.path.join(FIGS, "alert_lift_vs_rho.png"),
            order=["tcp", "tcp_m", "tcp_m_gate_tau", "kof_stab", "kof_stab_gate_tau", "kof_nostab", "red_lr"]
        )

    subF = fair_summary[
        (fair_summary["exp"] == "full") &
        (np.isclose(fair_summary["budget"], BSEL)) &
        (fair_summary["delta"] == -1.0)
    ].copy()
    if len(subF):
        for pol in ["global", "parity"]:
            dpol = subF[subF["policy"] == pol].copy()
            dpol = dpol[dpol["model"].isin(["tcp_m_gate_tau", "kof_stab_gate_tau", "kof_stab"])]
            if len(dpol):
                g_df = dpol[["rho", "model", "alert_rate_gap_mean", "alert_rate_gap_ci_low", "alert_rate_gap_ci_high"]].rename(
                    columns={"alert_rate_gap_mean": "mean", "alert_rate_gap_ci_low": "lo", "alert_rate_gap_ci_high": "hi"}
                )
                plot_line_ci(
                    g_df, "rho", "mean", "lo", "hi", "model",
                    f"Alert-rate gap vs rho @ budget={BSEL:.2f} ({pol})", "alert-rate gap (max-min gender)",
                    os.path.join(FIGS, f"fairness_gap_vs_rho_{pol}.png"),
                    order=["tcp_m_gate_tau", "kof_stab_gate_tau", "kof_stab"]
                )

                eod_df = dpol[["rho", "model", "equalized_odds_diff_mean", "equalized_odds_diff_ci_low", "equalized_odds_diff_ci_high"]].rename(
                    columns={"equalized_odds_diff_mean": "mean", "equalized_odds_diff_ci_low": "lo", "equalized_odds_diff_ci_high": "hi"}
                )
                plot_line_ci(
                    eod_df, "rho", "mean", "lo", "hi", "model",
                    f"Equalized odds difference vs rho @ budget={BSEL:.2f} ({pol})", "equalized odds diff (max TPR/FPR gap)",
                    os.path.join(FIGS, f"equalized_odds_diff_vs_rho_{pol}.png"),
                    order=["tcp_m_gate_tau", "kof_stab_gate_tau", "kof_stab"]
                )

                eoo_df = dpol[["rho", "model", "equal_opportunity_diff_mean", "equal_opportunity_diff_ci_low", "equal_opportunity_diff_ci_high"]].rename(
                    columns={"equal_opportunity_diff_mean": "mean", "equal_opportunity_diff_ci_low": "lo", "equal_opportunity_diff_ci_high": "hi"}
                )
                plot_line_ci(
                    eoo_df, "rho", "mean", "lo", "hi", "model",
                    f"Equal opportunity difference vs rho @ budget={BSEL:.2f} ({pol})", "equal opportunity diff (TPR gap)",
                    os.path.join(FIGS, f"equal_opportunity_diff_vs_rho_{pol}.png"),
                    order=["tcp_m_gate_tau", "kof_stab_gate_tau", "kof_stab"]
                )

    RHO_SEL = 1.0
    pareto = tradeoff_summary[
        (tradeoff_summary["exp"] == "full") &
        (np.isclose(tradeoff_summary["rho"], RHO_SEL)) &
        (np.isclose(tradeoff_summary["budget"], BSEL)) &
        (tradeoff_summary["model"].isin(["tcp", "tcp_m", "tcp_m_gate_tau", "kof_stab", "kof_stab_gate_tau"]))
    ].copy()
    if len(pareto):
        def _lab2(row):
            if row["policy"] == "bounded":
                return f"{row['model']}/bounded(d={row['delta']:.2f})"
            return f"{row['model']}/{row['policy']}"
        pareto["label"] = pareto.apply(_lab2, axis=1)

        plt.figure(figsize=(8, 6))
        for _, r in pareto.iterrows():
            x = float(r["alert_rate_gap_mean"])
            y = float(r["precision_mean"])
            xlo = float(r["alert_rate_gap_ci_low"])
            xhi = float(r["alert_rate_gap_ci_high"])
            ylo = float(r["precision_ci_low"])
            yhi = float(r["precision_ci_high"])
            plt.scatter([x], [y])
            plt.errorbar([x], [y], xerr=[[x - xlo], [xhi - x]], yerr=[[y - ylo], [yhi - y]], fmt="none", capsize=3)
            plt.text(x, y, str(r["label"]), fontsize=8, ha="left", va="bottom")
        plt.xlabel("alert-rate gap (max-min gender)")
        plt.ylabel("overall alert precision")
        plt.title(f"Pareto: precision vs fairness gap @ rho={int(RHO_SEL)}, budget={BSEL:.2f}")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(FIGS, f"pareto_precision_vs_gap_rho{int(RHO_SEL)}_budget{BSEL:.2f}.png"), dpi=160)
        plt.close()

    report_md = os.path.join(args.OUTDIR, "report.md")
    with open(report_md, "w", encoding="utf-8") as f:
        f.write("# Protocol-valid fair auditing of high-confidence failures under distribution shift (TABULAR)\n\n")
        f.write(f"Dataset: `{DATASET_NAME}` (UCI id=296; TableShift-like OOD split `admission_source_id == 7`). Sensitive attribute (eval-only): `{SENSITIVE_COL}` (Male=1 vs Female=0; Unknown/Invalid dropped)\n\n")
        f.write("## Framing\n")
        f.write(f"- Target label: z=1[err & conf>={args.TAU_TARGET:.2f}]\n")
        f.write(f"- Monitoring slice: conf>={args.HIGH_CONF_REGION:.2f}\n")
        f.write("- Allocation policies: global / parity / bounded(delta)\n")
        f.write("- Budgeting: K=floor(budget*n), allows K=0\n")
        f.write("- Lift: precision / P(z=1 | monitored slice)\n\n")

        f.write("## Versions\n```json\n")
        f.write(json.dumps(versions, indent=2))
        f.write("\n```\n\n")

        f.write("## Args\n```json\n")
        f.write(json.dumps(asdict(args), indent=2))
        f.write("\n```\n\n")

        f.write("## Summary: Overall (mean/std/CI)\n")
        f.write(overall_summary.to_markdown(index=False))
        f.write("\n\n")

        f.write("## Summary: Alerts (mean/std/CI) - includes lift and base-rate\n")
        f.write(alerts_summary.to_markdown(index=False))
        f.write("\n\n")

        base_tbl = alerts_summary[
            (alerts_summary["exp"] == "full") &
            (alerts_summary["model"] == "tcp_m") &
            (alerts_summary["policy"] == "global") &
            (alerts_summary["delta"] == -1.0) &
            (np.isclose(alerts_summary["budget"], BSEL))
        ][["rho", "z_rate_high_mean", "z_rate_high_ci_low", "z_rate_high_ci_high"]].copy()
        if len(base_tbl):
            f.write("## Base rate of z in monitored slice (conf>=HIGH_CONF_REGION)\n")
            f.write(base_tbl.to_markdown(index=False))
            f.write("\n\n")

        f.write("## Summary: Fairness (mean/std/CI)\n")
        f.write("Includes alert-rate gap/ratio (review-allocation disparity), precision gap, recall gap, "
                "FPR gap, Equal Opportunity Difference (max TPR gap across groups on the high-confidence-failure "
                "class), and Equalized Odds Difference (max of TPR gap and FPR gap across groups).\n\n")
        f.write(fair_summary.to_markdown(index=False))
        f.write("\n\n")

        if len(dataset_stats_summary):
            f.write("## Dataset and Shift Statistics\n")
            f.write("Dataset sizes after filtering, group distributions, class imbalance, number of "
                    "high-confidence failures, and number of positive target events per split / shift "
                    "condition (rho).\n\n")
            f.write(dataset_stats_summary.to_markdown(index=False))
            f.write("\n\n")

        f.write("## Summary: Tradeoff (mean/std/CI)\n")
        f.write(tradeoff_summary.to_markdown(index=False))
        f.write("\n\n")

        if len(raw_selchg):
            f.write("## Selection-only confirmation\n")
            f.write("Mitigation changes alert allocation only (scores unchanged). Rows show #changed alerts vs global.\n\n")
            f.write(raw_selchg.head(200).to_markdown(index=False))
            f.write("\n\n")

        f.write("## Figures\n")
        for fn in sorted(os.listdir(FIGS)):
            f.write(f"- figs/{fn}\n")

    zip_path = args.OUTDIR.rstrip("/").rstrip("\\") + ".zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(args.OUTDIR):
            for fn in files:
                full = os.path.join(root, fn)
                rel = os.path.relpath(full, args.OUTDIR)
                z.write(full, arcname=rel)

    script_out = os.path.join(args.OUTDIR, "tableshift_diabetes_common_monitor.py")
    try:
        with open(__file__, "r", encoding="utf-8") as rf:
            src = rf.read()
        with open(script_out, "w", encoding="utf-8") as wf:
            wf.write(src)
    except Exception:
        pass

    print("\nDONE.")
    print("OUTDIR:", args.OUTDIR)
    print("Zip:", zip_path)


if __name__ == "__main__":
    main()